In [1]:
import os
import json
import logging
import zipfile
import io
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time
from datetime import datetime
from collections import Counter

# --- Configuration ---
# Directories
BASE_DIR = "./OSRs"
NVD_DIR = os.path.join(BASE_DIR, "NVD")
MAPPING_DIR = os.path.join(BASE_DIR, "MAPPINGS")
LOG_FILE = "phase1_pipeline.log"
OUTPUT_FILE = "phase1_dataset_final.csv"

# URLs
# NVD 2.0 Bulk Feeds (Official NIST URL pattern)
NVD_URL_PATTERN = "https://nvd.nist.gov/feeds/json/cve/2.0/nvdcve-2.0-{year}.json.zip"
# CAPEC & CWE (XML is more reliable for transitive mapping)
CAPEC_URL = "https://capec.mitre.org/data/xml/capec_latest.xml"
CWE_URL = "https://cwe.mitre.org/data/xml/cwec_latest.xml.zip"
# CISA KEV (Gold Standard Labels)
KEV_URL = "https://center-for-threat-informed-defense.github.io/mappings-explorer/data/kev/attack-15.1/kev-02.13.2025/enterprise/kev-02.13.2025_attack-15.1-enterprise_json.json"

# Years to fetch from NVD
START_YEAR = 2002
END_YEAR = 2026

# --- Setup Logging ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, mode='w'), # Overwrite log each run
        logging.StreamHandler()
    ]
)

class CVEToATTACKPipeline:
    def __init__(self):
        self.capec_to_attack = {}  # CAPEC-ID -> Set(T-Codes)
        self.cwe_to_attack = {}    # CWE-ID -> Set(T-Codes)
        self.gold_labels = {}      # CVE-ID -> Set(T-Codes)
        self._ensure_directories()

    def _ensure_directories(self):
        for d in [NVD_DIR, MAPPING_DIR]:
            if not os.path.exists(d):
                os.makedirs(d)
                logging.info(f"Created directory: {d}")

    def download_nvd_data(self):
        """Downloads and extracts NVD JSON feeds (2002-2026)."""
        logging.info(f"--- [1] Checking NVD Data ({START_YEAR}-{END_YEAR}) ---")
        
        for year in range(START_YEAR, END_YEAR + 1):
            filename = f"nvdcve-2.0-{year}.json"
            filepath = os.path.join(NVD_DIR, filename)
            
            if os.path.exists(filepath):
                # Check if file is not empty/corrupt? For now, just exist check.
                continue

            url = NVD_URL_PATTERN.format(year=year)
            logging.info(f"Downloading NVD feed for {year}...")
            
            try:
                r = requests.get(url, stream=True)
                if r.status_code == 200:
                    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                        z.extractall(NVD_DIR)
                    logging.info(f"  -> Extracted {filename}")
                else:
                    logging.warning(f"  -> Failed to download {year} (Status {r.status_code})")
            except Exception as e:
                logging.error(f"  -> Error downloading {year}: {e}")
            
            time.sleep(1) # Be nice to NIST servers

    def download_mappings(self):
        """Downloads CAPEC (XML), CWE (Zip->XML), and KEV (JSON)."""
        logging.info("--- [2] Downloading Mapping Standards ---")
        
        # 1. CAPEC
        capec_path = os.path.join(MAPPING_DIR, "capec_latest.xml")
        if not os.path.exists(capec_path):
            logging.info(f"Downloading CAPEC from {CAPEC_URL}...")
            try:
                r = requests.get(CAPEC_URL)
                with open(capec_path, 'wb') as f:
                    f.write(r.content)
            except Exception as e:
                logging.error(f"Failed to download CAPEC: {e}")

        # 2. CWE (Robust Zip Extraction Logic)
        # We assume if cwec_latest.xml exists, we are good. If not, try download.
        final_cwe_path = os.path.join(MAPPING_DIR, "cwec_latest.xml")
        if not os.path.exists(final_cwe_path):
            logging.info(f"Downloading CWE from {CWE_URL}...")
            try:
                r = requests.get(CWE_URL)
                with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                    # Search for the .xml file inside the zip (name changes by version)
                    xml_files = [f for f in z.namelist() if f.endswith('.xml')]
                    if not xml_files:
                        logging.error("No XML found inside CWE zip!")
                    else:
                        target_file = xml_files[0]
                        logging.info(f"Extracting {target_file} from CWE Zip...")
                        z.extract(target_file, MAPPING_DIR)
                        
                        # Rename to standard name
                        extracted_path = os.path.join(MAPPING_DIR, target_file)
                        if os.path.exists(final_cwe_path):
                            os.remove(final_cwe_path)
                        os.rename(extracted_path, final_cwe_path)
            except Exception as e:
                logging.error(f"Failed to download/extract CWE: {e}")

        # 3. KEV (Gold Standard)
        kev_path = os.path.join(MAPPING_DIR, "kev_mapping.json")
        if not os.path.exists(kev_path):
            logging.info(f"Downloading KEV from {KEV_URL}...")
            try:
                r = requests.get(KEV_URL)
                with open(kev_path, 'wb') as f:
                    f.write(r.content)
            except Exception as e:
                logging.error(f"Failed to download KEV: {e}")

    def parse_mappings(self):
        """Parses the downloaded files to build the dictionaries (Namespace Agnostic)."""
        logging.info("--- [3] Parsing Standards (XML/JSON) ---")
        
        # Helper to strip namespaces like {http://capec.mitre.org...}
        strip_ns = lambda t: t.split('}', 1)[1] if '}' in t else t
        
        # 1. CAPEC Parsing
        capec_path = os.path.join(MAPPING_DIR, "capec_latest.xml")
        try:
            tree = ET.parse(capec_path)
            root = tree.getroot()
            
            for elem in root.iter():
                if strip_ns(elem.tag) == "Attack_Pattern":
                    capec_id = elem.get("ID")
                    if not capec_id: continue
                    full_capec = f"CAPEC-{capec_id}"
                    
                    # Look for Taxonomy_Mappings -> ATTACK
                    for child in elem.iter():
                        if strip_ns(child.tag) == "Taxonomy_Mapping":
                            if child.get("Taxonomy_Name") == "ATTACK":
                                for sub in child:
                                    if strip_ns(sub.tag) == "Entry_ID":
                                        tid = sub.text.strip()
                                        if not tid.startswith("T"): tid = "T" + tid
                                        
                                        if full_capec not in self.capec_to_attack:
                                            self.capec_to_attack[full_capec] = set()
                                        self.capec_to_attack[full_capec].add(tid)
            logging.info(f"Mapped {len(self.capec_to_attack)} CAPECs to ATT&CK.")
        except Exception as e:
            logging.error(f"CAPEC Parse Error: {e}")

        # 2. CWE Parsing
        cwe_path = os.path.join(MAPPING_DIR, "cwec_latest.xml")
        try:
            tree = ET.parse(cwe_path)
            root = tree.getroot()
            for elem in root.iter():
                if strip_ns(elem.tag) == "Weakness":
                    cwe_id = elem.get("ID")
                    if not cwe_id: continue
                    full_cwe = f"CWE-{cwe_id}"
                    
                    for child in elem.iter():
                        if strip_ns(child.tag) == "Related_Attack_Pattern":
                            ref = child.get("CAPEC_ID")
                            if ref:
                                full_ref = f"CAPEC-{ref}"
                                if full_ref in self.capec_to_attack:
                                    if full_cwe not in self.cwe_to_attack:
                                        self.cwe_to_attack[full_cwe] = set()
                                    self.cwe_to_attack[full_cwe].update(self.capec_to_attack[full_ref])
            logging.info(f"Mapped {len(self.cwe_to_attack)} CWEs to ATT&CK.")
        except Exception as e:
            logging.error(f"CWE Parse Error: {e}")

        # 3. KEV Parsing
        kev_path = os.path.join(MAPPING_DIR, "kev_mapping.json")
        try:
            with open(kev_path, 'r') as f:
                data = json.load(f)
            for obj in data.get('mapping_objects', []):
                cve = obj.get('capability_id')
                tech = obj.get('attack_object_id')
                if cve and tech and cve.startswith("CVE-"):
                    if cve not in self.gold_labels:
                        self.gold_labels[cve] = set()
                    self.gold_labels[cve].add(tech)
            logging.info(f"Loaded {len(self.gold_labels)} Gold KEV Labels.")
        except Exception as e:
            logging.error(f"KEV Parse Error: {e}")

    def generate_dataset(self):
        """Scans NVD files and applies the maps."""
        logging.info("--- [4] Building Final Dataset ---")
        
        files = [f for f in os.listdir(NVD_DIR) if f.endswith('.json')]
        logging.info(f"Scanning {len(files)} NVD files...")
        
        dataset = []
        
        for file in files:
            path = os.path.join(NVD_DIR, file)
            try:
                with open(path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                for item in data.get('vulnerabilities', []):
                    cve_item = item.get('cve', {})
                    cve_id = cve_item.get('id')
                    
                    # Get Text
                    desc = "N/A"
                    for d in cve_item.get('descriptions', []):
                        if d.get('lang') == 'en':
                            desc = d.get('value')
                            break
                    if desc == "N/A": continue
                    
                    labels = set()
                    sources = []

                    # A. Gold Labels
                    if cve_id in self.gold_labels:
                        labels.update(self.gold_labels[cve_id])
                        sources.append("gold")

                    # B. Transitive Labels (CWE)
                    cwe_list = []
                    for w in cve_item.get('weaknesses', []):
                        for d in w.get('description', []):
                            val = d.get('value')
                            if val.startswith("CWE-"):
                                cwe_list.append(val)
                    
                    for cwe in cwe_list:
                        if cwe in self.cwe_to_attack:
                            labels.update(self.cwe_to_attack[cwe])
                            if "transitive" not in sources:
                                sources.append("transitive")

                    if labels:
                        dataset.append({
                            'cve_id': cve_id,
                            'text': desc,
                            'labels': list(labels),
                            'source': "+".join(sources)
                        })

            except Exception as e:
                logging.warning(f"Skipping file {file}: {e}")

        # Save
        df = pd.DataFrame(dataset)
        df.to_csv(OUTPUT_FILE, index=False)
        logging.info(f"Dataset saved to {OUTPUT_FILE}")
        return df

    def validate_output(self, df):
        """Prints statistical validation of the dataset."""
        logging.info("--- [5] Validation & Stats ---")
        
        report = []
        report.append("\n" + "="*30)
        report.append("DATASET VALIDATION REPORT")
        report.append("="*30)
        report.append(f"Total Samples: {len(df)}")
        
        # 1. Source Distribution
        report.append("\n[Source Distribution]")
        report.append(str(df['source'].value_counts()))
        
        # 2. Label Stats
        all_labels = []
        for row in df['labels']:
            try:
                # If pandas loaded it as a string representation of a list
                lbls = eval(row) if isinstance(row, str) else row
                all_labels.extend(lbls)
            except:
                pass
                
        report.append(f"\n[Label Stats]")
        report.append(f"Unique Techniques Found: {len(set(all_labels))}")
        report.append(f"Total Assignments: {len(all_labels)}")
        
        # 3. Top 5 Techniques
        report.append("\n[Top 5 Most Common Techniques]")
        counts = Counter(all_labels)
        for tech, count in counts.most_common(5):
            report.append(f"  {tech}: {count} samples")

        report.append("="*30 + "\n")
        
        # Print to console and log
        final_report = "\n".join(report)
        print(final_report)
        logging.info(final_report)

if __name__ == "__main__":
    pipeline = CVEToATTACKPipeline()
    pipeline.download_nvd_data()
    pipeline.download_mappings()
    pipeline.parse_mappings()
    df = pipeline.generate_dataset()
    pipeline.validate_output(df)

2026-04-05 02:08:07,880 [INFO] --- [1] Checking NVD Data (2002-2026) ---
2026-04-05 02:08:07,880 [INFO] --- [2] Downloading Mapping Standards ---
2026-04-05 02:08:07,880 [INFO] --- [3] Parsing Standards (XML/JSON) ---
2026-04-05 02:08:07,945 [INFO] Mapped 177 CAPECs to ATT&CK.
2026-04-05 02:08:08,445 [INFO] Mapped 149 CWEs to ATT&CK.
2026-04-05 02:08:08,464 [INFO] Loaded 296 Gold KEV Labels.
2026-04-05 02:08:08,479 [INFO] --- [4] Building Final Dataset ---
2026-04-05 02:08:08,479 [INFO] Scanning 25 NVD files...
2026-04-05 02:08:42,228 [INFO] Dataset saved to phase1_dataset_final.csv
2026-04-05 02:08:42,312 [INFO] --- [5] Validation & Stats ---
2026-04-05 02:08:42,362 [INFO] 
DATASET VALIDATION REPORT
Total Samples: 77521

[Source Distribution]
source
transitive         77225
gold                 207
gold+transitive       89
Name: count, dtype: int64

[Label Stats]
Unique Techniques Found: 250
Total Assignments: 660495

[Top 5 Most Common Techniques]
  T1574.007: 28363 samples
  T1574.0


DATASET VALIDATION REPORT
Total Samples: 77521

[Source Distribution]
source
transitive         77225
gold                 207
gold+transitive       89
Name: count, dtype: int64

[Label Stats]
Unique Techniques Found: 250
Total Assignments: 660495

[Top 5 Most Common Techniques]
  T1574.007: 28363 samples
  T1574.006: 26946 samples
  T1562.003: 26820 samples
  T1027: 16258 samples
  T1083: 14563 samples



In [2]:
import pandas as pd
import ast
import re

# --- Configuration ---
INPUT_FILE = "phase1_dataset_final.csv"
OUTPUT_FILE = "phase1_dataset_parent_only.csv"

def convert_to_parent(label_list):
    """
    Input:  ['T1059.001', 'T1059.006', 'T1003']
    Output: ['T1059', 'T1003']
    """
    if isinstance(label_list, str):
        label_list = ast.literal_eval(label_list)
        
    parent_labels = set()
    for label in label_list:
        # Regex to capture just the Txxxx part, ignoring .yyy
        # Matches T followed by digits, stops at dot or end
        match = re.match(r"(T\d+)", label)
        if match:
            parent_labels.add(match.group(1))
            
    return list(parent_labels)

def main():
    print(f"[*] Loading {INPUT_FILE}...")
    df = pd.read_csv(INPUT_FILE)
    
    # Apply transformation
    print("[*] Merging Sub-Techniques -> Parent Techniques...")
    df['labels'] = df['labels'].apply(convert_to_parent)
    
    # Remove rows that became empty (rare, but possible if a label was malformed)
    df = df[df['labels'].map(len) > 0]
    
    # Save
    print(f"[*] Saving to {OUTPUT_FILE}...")
    df.to_csv(OUTPUT_FILE, index=False)
    
    # Stats
    all_labels = [lbl for row in df['labels'] for lbl in row]
    unique_parents = len(set(all_labels))
    print(f"\n[SUCCESS]")
    print(f"New Class Count: {unique_parents} Parent Techniques")
    print(f"Total Samples:   {len(df)}")
    print("You can now re-run Phase 2 with this new file.")

if __name__ == "__main__":
    main()

[*] Loading phase1_dataset_final.csv...
[*] Merging Sub-Techniques -> Parent Techniques...
[*] Saving to phase1_dataset_parent_only.csv...

[SUCCESS]
New Class Count: 134 Parent Techniques
Total Samples:   77521
You can now re-run Phase 2 with this new file.


In [3]:
import pandas as pd
import numpy as np
import re
import ast
import pickle
import logging
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from collections import Counter

# --- Configuration ---
INPUT_FILE = "phase1_dataset_parent_only.csv"
OUTPUT_DIR = "./processed_data_parent"
MIN_SAMPLES_PER_CLASS = 10  # Drop techniques with fewer than X examples
TEST_SIZE = 0.2
RANDOM_STATE = 420

# Setup Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

class Phase2Preprocessor:
    def __init__(self):
        self.df = None
        self.mlb = MultiLabelBinarizer()
        
    def load_data(self):
        logging.info(f"Loading data from {INPUT_FILE}...")
        if not os.path.exists(INPUT_FILE):
            raise FileNotFoundError(f"{INPUT_FILE} not found. Did Phase 1 finish?")
        
        self.df = pd.read_csv(INPUT_FILE)
        
        # Convert string representation of lists "['T1', 'T2']" back to actual lists
        self.df['labels'] = self.df['labels'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
        
        logging.info(f"Loaded {len(self.df)} rows.")

    def clean_text(self, text):
        """
        Removes URLs, CVE IDs, hex codes, and extra whitespace.
        We keep the text mostly intact for Contextual Embeddings (BERT) later,
        but remove specific artifacts that cause overfitting.
        """
        if not isinstance(text, str):
            return ""
        
        # 1. Lowercase
        text = text.lower()
        
        # 2. Remove CVE IDs (Data Leakage Prevention)
        # e.g., "CVE-2021-44228" -> "vulnerability"
        text = re.sub(r'cve-\d{4}-\d{4,7}', 'vulnerability', text)
        
        # 3. Remove URLs
        text = re.sub(r'http\S+|www\.\S+', '', text)
        
        # 4. Remove Hex Codes / Memory Addresses (common in exploits)
        # e.g., 0x41414141
        text = re.sub(r'0x[a-f0-9]+', '', text)
        
        # 5. Remove special chars but keep hyphens/dots (useful for versions)
        text = re.sub(r'[^a-z0-9\s\-\.]', '', text)
        
        # 6. Normalize whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text

    def preprocess_text(self):
        logging.info("Cleaning text descriptions...")
        # Create a clean column, keep original for reference
        self.df['clean_text'] = self.df['text'].apply(self.clean_text)
        
        # Drop rows that became empty after cleaning
        initial_len = len(self.df)
        self.df = self.df[self.df['clean_text'].str.len() > 10]
        logging.info(f"Dropped {initial_len - len(self.df)} empty/short rows.")

    def filter_rare_classes(self):
        """
        Implements the 'Balancing' strategy.
        Removes labels that appear fewer than MIN_SAMPLES_PER_CLASS times.
        """
        logging.info(f"Filtering rare classes (Threshold: {MIN_SAMPLES_PER_CLASS})...")
        
        # Flatten all labels to count frequency
        all_labels = [label for sublist in self.df['labels'] for label in sublist]
        counts = Counter(all_labels)
        
        # Identify valid labels
        valid_labels = {lbl for lbl, count in counts.items() if count >= MIN_SAMPLES_PER_CLASS}
        dropped_labels = {lbl for lbl, count in counts.items() if count < MIN_SAMPLES_PER_CLASS}
        
        logging.info(f"Found {len(valid_labels)} valid techniques. Dropping {len(dropped_labels)} rare ones.")
        
        # Filter the label lists in the dataframe
        def filter_row_labels(label_list):
            return [lbl for lbl in label_list if lbl in valid_labels]
        
        self.df['filtered_labels'] = self.df['labels'].apply(filter_row_labels)
        
        # Drop rows that now have NO labels (because all their labels were rare)
        initial_len = len(self.df)
        self.df = self.df[self.df['filtered_labels'].map(len) > 0]
        logging.info(f"Dropped {initial_len - len(self.df)} rows that lost all labels after filtering.")

    def encode_labels(self):
        """
        Converts lists of strings to a binary matrix.
        ['T1', 'T2'] -> [1, 1, 0, 0, ...]
        """
        logging.info("Binarizing labels (One-Hot Encoding)...")
        
        # Fit MultiLabelBinarizer
        y = self.mlb.fit_transform(self.df['filtered_labels'])
        
        # Get class names
        classes = self.mlb.classes_
        logging.info(f"Final Class Count: {len(classes)}")
        
        return y, classes

    def save_artifacts(self, y):
        if not os.path.exists(OUTPUT_DIR):
            os.makedirs(OUTPUT_DIR)
            
        logging.info("Splitting Train/Test and saving artifacts...")
        
        # Split Data
        X = self.df['clean_text'].values
        X = np.array(X)
        y = np.array(y)
        
        # We use a standard random split here. 
        # (For strict multi-label stratification, 'iterative-stratification' lib is better, 
        # but standard split is usually fine for N > 5000).
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
        )
        
        # Save Metadata (Class Names)
        with open(os.path.join(OUTPUT_DIR, "mlb.pkl"), "wb") as f:
            pickle.dump(self.mlb, f)
            
        # Save Datasets (Parquet is faster/smaller than CSV for arrays, but CSV is human readable. 
        # We will save as Numpy arrays for the matrices and CSV for text)
        
        # Save Text
        pd.DataFrame(X_train, columns=['text']).to_csv(os.path.join(OUTPUT_DIR, "train_text.csv"), index=False)
        pd.DataFrame(X_test, columns=['text']).to_csv(os.path.join(OUTPUT_DIR, "test_text.csv"), index=False)
        
        # Save Label Matrices (Compressed Numpy)
        np.savez_compressed(os.path.join(OUTPUT_DIR, "y_train.npz"), y_train)
        np.savez_compressed(os.path.join(OUTPUT_DIR, "y_test.npz"), y_test)
        
        logging.info(f"Processing Complete.")
        logging.info(f"Train Size: {len(X_train)}")
        logging.info(f"Test Size: {len(X_test)}")
        logging.info(f"Outputs saved to {OUTPUT_DIR}/")

if __name__ == "__main__":
    import os
    
    preprocessor = Phase2Preprocessor()
    preprocessor.load_data()
    preprocessor.preprocess_text()
    preprocessor.filter_rare_classes()
    y_matrix, class_names = preprocessor.encode_labels()
    preprocessor.save_artifacts(y_matrix)

2026-04-05 02:08:44,728 [INFO] Loading data from phase1_dataset_parent_only.csv...
2026-04-05 02:08:45,578 [INFO] Loaded 77521 rows.
2026-04-05 02:08:45,578 [INFO] Cleaning text descriptions...
2026-04-05 02:08:46,562 [INFO] Dropped 0 empty/short rows.
2026-04-05 02:08:46,562 [INFO] Filtering rare classes (Threshold: 10)...
2026-04-05 02:08:46,612 [INFO] Found 105 valid techniques. Dropping 29 rare ones.
2026-04-05 02:08:46,795 [INFO] Dropped 4 rows that lost all labels after filtering.
2026-04-05 02:08:46,795 [INFO] Binarizing labels (One-Hot Encoding)...
2026-04-05 02:08:46,862 [INFO] Final Class Count: 105
2026-04-05 02:08:46,878 [INFO] Splitting Train/Test and saving artifacts...
2026-04-05 02:08:47,462 [INFO] Processing Complete.
2026-04-05 02:08:47,462 [INFO] Train Size: 62013
2026-04-05 02:08:47,462 [INFO] Test Size: 15504
2026-04-05 02:08:47,462 [INFO] Outputs saved to ./processed_data_parent/


In [4]:
import pandas as pd
import numpy as np
import pickle
import os
import json
import ast
import re
import logging
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    coverage_error, 
    label_ranking_loss, 
    label_ranking_average_precision_score
)
from sklearn.pipeline import Pipeline
import joblib

# --- Configuration ---
DATA_DIR = "./processed_data_parent"
MODEL_DIR = "./models/baseline_parent"
MAX_FEATURES = 5000  # Vocabulary size (Top 5k words)

# Setup Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

class Phase3Baseline:
    def __init__(self):
        self.X_train = None
        self.y_train = None
        self.classes = None
        self.model = None

    def load_training_data(self):
        logging.info("Loading preprocessed training data...")
        
        # Load Training Text
        self.X_train = pd.read_csv(os.path.join(DATA_DIR, "train_text.csv"))['text'].fillna("").values
        
        # Load Training Labels
        self.y_train = np.load(os.path.join(DATA_DIR, "y_train.npz"))['arr_0']
        
        # Load Class Names
        with open(os.path.join(DATA_DIR, "mlb.pkl"), "rb") as f:
            mlb = pickle.load(f)
            self.classes = mlb.classes_
            
        logging.info(f"Training Data Loaded. Classes: {len(self.classes)}")

    def train_baseline(self):
        logging.info("Initializing TF-IDF + Logistic Regression Pipeline...")
        self.model = Pipeline([
            ('tfidf', TfidfVectorizer(max_features=MAX_FEATURES, stop_words='english')),
            ('clf', OneVsRestClassifier(LogisticRegression(solver='liblinear', class_weight='balanced')))
        ])
        
        logging.info("Training Model (this may take 1-2 minutes)...")
        self.model.fit(self.X_train, self.y_train)
        logging.info("Training Complete.")

    def evaluate_on_cve_benchmark(self):
        """Evaluates the model specifically on the CVE Benchmark Dataset like SMET"""
        logging.info("Loading ATT&CK STIX data & Benchmark Data...")
        
        # 1. Load Mappings
        try:
            id2mitre = json.load(open('id2mitre.json', 'r'))
            with open(r"OSRs\ATTACK\enterprise-attack-v16.1.json", "r", encoding="utf-8") as f:
                stix_bundle = json.load(f)
        except Exception as e:
            logging.error(f"Could not load JSON mapping files: {e}")
            return

        name_to_tcode = {}
        parent_map = {}

        # 2. Extract STIX Relationships
        for obj in stix_bundle.get("objects", []):
            if obj.get("type") == "attack-pattern" and not obj.get("revoked") and not obj.get("x_mitre_deprecated"):
                t_code = next((ref.get("external_id") for ref in obj.get("external_references", []) if ref.get("source_name") == "mitre-attack"), None)
                if t_code:
                    name_to_tcode[obj.get("name")] = t_code

        for obj in stix_bundle.get("objects", []):
            if obj.get("type") == "relationship" and obj.get("relationship_type") == "subtechnique-of":
                sub_id, parent_id = obj.get("source_ref"), obj.get("target_ref")
                sub_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == sub_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
                parent_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == parent_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
                if sub_tcode and parent_tcode:
                    parent_map[sub_tcode] = parent_tcode

        # 3. Load CVE Dataset
        smet_raw = pd.read_excel("CVE_annotated_dataset.xlsx")
        smet_records = []
        for _, row in smet_raw.iterrows():
            try:
                tech_names = ast.literal_eval(row["ATT&CK Techniques"])
            except:
                tech_names = []
                
            t_codes = []
            for name in tech_names:
                mapped_id = next((k for k, v in id2mitre.items() if name in v), name_to_tcode.get(name))
                if mapped_id:
                    t_codes.append(parent_map.get(mapped_id, mapped_id))
                    
            if t_codes:
                smet_records.append({
                    "Description": row["Description"],
                    "T_Codes": list(set(t_codes))
                })
                
        smet_df = pd.DataFrame(smet_records)
        cve_texts = smet_df['Description'].tolist()
        cve_ground_truths = smet_df['T_Codes'].tolist()

        # 4. Define evaluation space based on the baseline model's known classes
        # Assuming self.classes contains T-Codes (or names that map to T-Codes)
        baseline_parent_classes = sorted(list(set([parent_map.get(t, t) for t in self.classes])))
        num_classes = len(baseline_parent_classes)
        
        logging.info(f"Total Parent Classes handled by Baseline: {num_classes}")
        logging.info("Running full dataset evaluation on CVE Benchmark...")

        baseline_metrics = {'R@1': [], 'R@5': [], 'R@10': []}
        y_true_all = []
        y_score_all = []
        valid_cves = 0

        # 5. Full Dataset Evaluation Loop
        for idx, cve_desc in enumerate(tqdm(cve_texts, desc="Evaluating Pure Baseline")):
            ground_truth = set(cve_ground_truths[idx])
            n_true = len(ground_truth)
            
            if n_true == 0:
                continue
                
            valid_cves += 1

            # Get probabilities from Baseline
            probs = self.model.predict_proba([cve_desc])[0]
            
            # Roll up Baseline predictions to parents
            parent_probs = {tid: 0.0 for tid in baseline_parent_classes}
            for i, class_label in enumerate(self.classes):
                parent_tc = parent_map.get(class_label, class_label)
                if parent_tc in parent_probs:
                    parent_probs[parent_tc] = max(parent_probs[parent_tc], probs[i])

            # Populate Metric Matrices
            sample_y_true = np.zeros(num_classes)
            sample_y_score = np.zeros(num_classes)
            
            for i, tid in enumerate(baseline_parent_classes):
                sample_y_score[i] = parent_probs[tid]
                if tid in ground_truth:
                    sample_y_true[i] = 1.0
                    
            y_true_all.append(sample_y_true)
            y_score_all.append(sample_y_score)
            
            # Calculate Recall @ K
            sorted_indices = np.argsort(sample_y_score)[::-1]
            sorted_tcodes = [baseline_parent_classes[i] for i in sorted_indices]
                    
            for k in [1, 5, 10]:
                s_hits = len(set(sorted_tcodes[:k]).intersection(ground_truth))
                baseline_metrics[f'R@{k}'].append(s_hits / n_true)

        # Convert arrays for sklearn
        y_true_all = np.array(y_true_all)
        y_score_all = np.array(y_score_all)

        # Calculate Advanced Metrics
        lrap = label_ranking_average_precision_score(y_true_all, y_score_all)
        rl = label_ranking_loss(y_true_all, y_score_all)
        ce = coverage_error(y_true_all, y_score_all)

        # Print Final Averaged Results
        print("\n" + "="*50)
        print(f"BASELINE EVALUATION RESULTS (Tested on {valid_cves} CVEs)")
        print("="*50)
        print(f"Recall@1:       {(np.mean(baseline_metrics['R@1']) * 100):.2f}%")
        print(f"Recall@5:       {(np.mean(baseline_metrics['R@5']) * 100):.2f}%")
        print(f"Recall@10:      {(np.mean(baseline_metrics['R@10']) * 100):.2f}%")
        print("-" * 50)
        print(f"LRAP:           {lrap * 100:.2f}%  (Higher is better)")
        print(f"Ranking Loss:   {rl:.4f}   (Lower is better)")
        print(f"Coverage Error: {ce:.4f}   (Lower is better)")
        print("="*50)

    def save_model(self):
        if not os.path.exists(MODEL_DIR):
            os.makedirs(MODEL_DIR)
            
        model_path = os.path.join(MODEL_DIR, "tfidf_logreg.joblib")
        joblib.dump(self.model, model_path)
        logging.info(f"Model saved to {model_path}")

if __name__ == "__main__":
    baseline = Phase3Baseline()
    baseline.load_training_data()
    baseline.train_baseline()
    
    # Run the Benchmark Evaluation rather than the standard split
    baseline.evaluate_on_cve_benchmark()
    
    baseline.save_model()

c:\Users\OA\Desktop\New Work\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-05 02:00:15,553 [INFO] Loading preprocessed training data...
2026-04-05 02:00:15,935 [INFO] Training Data Loaded. Classes: 105
2026-04-05 02:00:15,935 [INFO] Initializing TF-IDF + Logistic Regression Pipeline...
2026-04-05 02:00:15,937 [INFO] Training Model (this may take 1-2 minutes)...
2026-04-05 02:01:10,829 [INFO] Training Complete.
2026-04-05 02:01:10,829 [INFO] Loading ATT&CK STIX data & Benchmark Data...
2026-04-05 02:01:11,242 [INFO] Total Parent Classes handled by Baseline: 105
2026-04-05 02:01:11,242 [INFO] Running full dataset evaluation on CVE Benchmark...
Evaluating Pure Baseline: 100%|██████████| 302/302 [00:01<00:00, 207.73it/s]
2026-04-05 02:01:12,813 [INFO] Model saved to ./models/baseline_parent\tfidf_logre


BASELINE EVALUATION RESULTS (Tested on 302 CVEs)
Recall@1:       6.13%
Recall@5:       16.53%
Recall@10:      25.63%
--------------------------------------------------
LRAP:           17.59%  (Higher is better)
Ranking Loss:   0.4359   (Lower is better)
Coverage Error: 52.9834   (Lower is better)


# AttackBert Exp



In [4]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
import logging
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.utils.tensorboard import SummaryWriter
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, label_ranking_average_precision_score

# --- Configuration ---
DATA_DIR = "./processed_data_parent"
MODEL_DIR = "./models/attackBert_parent" 
LOG_DIR = "./runs/attackBert_experiment_parent"
MODEL_NAME = "basel/ATTACK-BERT"            
MAX_LEN = 256
BATCH_SIZE = 64   
EPOCHS = 10
LEARNING_RATE = 2e-5
LOG_INTERVAL = 1  

# Setup Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
writer = SummaryWriter(LOG_DIR)

if torch.cuda.is_available():
    device = torch.device('cuda')
    logging.info(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    logging.warning("GPU not detected! Training will be slow.")

class CVEDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',     
            truncation=True,          
            return_attention_mask=True,
            return_tensors='pt',      
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.FloatTensor(label)
        }

def load_data():
    logging.info("Loading Data...")
    train_text = pd.read_csv(os.path.join(DATA_DIR, "train_text.csv"))['text'].fillna("").values
    test_text = pd.read_csv(os.path.join(DATA_DIR, "test_text.csv"))['text'].fillna("").values
    y_train = np.load(os.path.join(DATA_DIR, "y_train.npz"))['arr_0']
    y_test = np.load(os.path.join(DATA_DIR, "y_test.npz"))['arr_0']
    
    with open(os.path.join(DATA_DIR, "mlb.pkl"), "rb") as f:
        mlb = pickle.load(f)
        
    return train_text, test_text, y_train, y_test, len(mlb.classes_)

def train_epoch(model, data_loader, loss_fn, optimizer, scheduler, device, n_examples):
    model = model.train()
    losses = []
    
    total_steps = len(data_loader)
    for i, d in enumerate(data_loader):
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        targets = d["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        logits = outputs.logits
        loss = loss_fn(logits, targets)

        losses.append(loss.item())
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        if (i + 1) % LOG_INTERVAL == 0:
            avg_loss = np.mean(losses[-LOG_INTERVAL:])
            global_step = (n_examples * total_steps) + i
            logging.info(f"Epoch [{n_examples+1}/{EPOCHS}] Step [{i+1}/{total_steps}] Loss: {avg_loss:.4f}")
            writer.add_scalar('Training Loss', avg_loss, global_step)

    return np.mean(losses)

def eval_model(model, data_loader, loss_fn, device):
    model = model.eval()
    losses = []
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            loss = loss_fn(outputs.logits, targets)
            losses.append(loss.item())
            
            probs = torch.sigmoid(outputs.logits).cpu().numpy()
            targets = targets.cpu().numpy()
            
            all_preds.extend(probs)
            all_targets.extend(targets)

    return np.mean(losses), np.array(all_preds), np.array(all_targets)

def main():
    X_train, X_test, y_train, y_test, n_classes = load_data()
    
    logging.info(f"Loading Tokenizer ({MODEL_NAME})...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    train_ds = CVEDataset(X_train, y_train, tokenizer, MAX_LEN)
    test_ds = CVEDataset(X_test, y_test, tokenizer, MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=0)
    
    logging.info(f"Initializing Model (Classes: {n_classes})...")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, 
        num_labels=n_classes,
        problem_type="multi_label_classification"
    )
    model = model.to(device)
    
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    
    loss_fn = torch.nn.BCEWithLogitsLoss()
    
    logging.info("Starting Training with ATTACK-BERT...")
    
    # Track the BEST LRAP instead of F1
    best_lrap = 0.0 
    
    if not os.path.exists(MODEL_DIR):
        os.makedirs(MODEL_DIR)

    for epoch in range(EPOCHS):
        logging.info(f"Epoch {epoch + 1}/{EPOCHS}")
        
        train_loss = train_epoch(
            model, train_loader, loss_fn, optimizer, scheduler, device, epoch
        )
        
        logging.info(f"Train loss: {train_loss:.4f}")
        
        val_loss, preds, targets = eval_model(
            model, test_loader, loss_fn, device
        )
        
        # Calculate standard F1 just for logging
        preds_binary = (preds > 0.5).astype(int)
        val_f1 = f1_score(targets, preds_binary, average='micro')
        
        # Calculate LRAP for actual model selection!
        val_lrap = label_ranking_average_precision_score(targets, preds)
        
        logging.info(f"Val loss: {val_loss:.4f} | Micro F1: {val_f1:.4f} | LRAP: {val_lrap:.4f}")
        writer.add_scalar('Validation Loss', val_loss, epoch)
        writer.add_scalar('Validation F1', val_f1, epoch)     
        writer.add_scalar('Validation LRAP', val_lrap, epoch)

        # Save model based on LRAP
        if val_lrap > best_lrap:
            logging.info(f"New Best LRAP! ({val_lrap:.4f} > {best_lrap:.4f}). Saving model...")            
            model.save_pretrained(MODEL_DIR)
            tokenizer.save_pretrained(MODEL_DIR)
            best_lrap = val_lrap

    logging.info(f"Training Complete. Best Validation LRAP: {best_lrap:.4f}")
    writer.close()

if __name__ == "__main__":
    main()

c:\Users\OA\Desktop\New Work\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-05 02:08:50,929 [INFO] Using GPU: NVIDIA GeForce RTX 3090
2026-04-05 02:08:50,929 [INFO] Loading Data...
2026-04-05 02:08:51,245 [INFO] Loading Tokenizer (basel/ATTACK-BERT)...
2026-04-05 02:08:51,345 [INFO] HTTP Request: HEAD https://huggingface.co/basel/ATTACK-BERT/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-05 02:08:51,345 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-05 02:08:51,362 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/basel/ATTACK-BERT/81e30f983a822c606825507b63ae318a6830a8a2/config.json "HTTP/1.1 200 OK"
2026-04-05 02:08:51,405 [INFO] HTTP Request: HEA

In [5]:
import torch
import numpy as np
import pandas as pd
import pickle
import logging
import os
import json
import ast
import re
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    coverage_error, 
    label_ranking_loss, 
    label_ranking_average_precision_score
)

# --- Configuration ---
DATA_DIR = "./processed_data_parent"
MODEL_DIR = "./models/attackBert_parent"
BATCH_SIZE = 32
MAX_LEN = 256

# Setup Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
    logging.warning("Using CPU. Inference might be slow.")

class EvaluationDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

def get_predictions(model, loader):
    model.eval()
    all_probs = []
    
    logging.info("Running Inference on CVE Benchmark Dataset...")
    with torch.no_grad():
        for d in tqdm(loader, desc="BERT Batches"):
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits)
            all_probs.extend(probs.cpu().numpy())
            
    return np.array(all_probs)

def main():
    # 1. Load the BERT Model and Class Names (MLB)
    logging.info("Loading Fine-Tuned BERT model and Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
    model.to(device)
    
    with open(os.path.join(DATA_DIR, "mlb.pkl"), "rb") as f:
        mlb = pickle.load(f)
        bert_classes = mlb.classes_
        
    logging.info(f"Model loaded. Output layer size: {len(bert_classes)}")

    # 2. Load Mappings (STIX & id2mitre)
    logging.info("Loading ATT&CK STIX data & Benchmark Data...")
    try:
        id2mitre = json.load(open('id2mitre.json', 'r'))
        with open(r"OSRs\ATTACK\enterprise-attack-v16.1.json", "r", encoding="utf-8") as f:
            stix_bundle = json.load(f)
    except Exception as e:
        logging.error(f"Could not load JSON mapping files: {e}")
        return

    name_to_tcode = {}
    parent_map = {}

    for obj in stix_bundle.get("objects", []):
        if obj.get("type") == "attack-pattern" and not obj.get("revoked") and not obj.get("x_mitre_deprecated"):
            t_code = next((ref.get("external_id") for ref in obj.get("external_references", []) if ref.get("source_name") == "mitre-attack"), None)
            if t_code:
                name_to_tcode[obj.get("name")] = t_code

    for obj in stix_bundle.get("objects", []):
        if obj.get("type") == "relationship" and obj.get("relationship_type") == "subtechnique-of":
            sub_id, parent_id = obj.get("source_ref"), obj.get("target_ref")
            sub_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == sub_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
            parent_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == parent_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
            if sub_tcode and parent_tcode:
                parent_map[sub_tcode] = parent_tcode

    # 3. Load CVE Dataset
    smet_raw = pd.read_excel("CVE_annotated_dataset.xlsx")
    smet_records = []
    for _, row in smet_raw.iterrows():
        try:
            tech_names = ast.literal_eval(row["ATT&CK Techniques"])
        except:
            tech_names = []
            
        t_codes = []
        for name in tech_names:
            mapped_id = next((k for k, v in id2mitre.items() if name in v), name_to_tcode.get(name))
            if mapped_id:
                t_codes.append(parent_map.get(mapped_id, mapped_id))
                
        if t_codes:
            smet_records.append({
                "Description": row["Description"],
                "T_Codes": list(set(t_codes))
            })
            
    smet_df = pd.DataFrame(smet_records)
    cve_texts = smet_df['Description'].tolist()
    cve_ground_truths = smet_df['T_Codes'].tolist()

    # 4. Prepare Data Loader for CVE texts
    dataset = EvaluationDataset(cve_texts, tokenizer, MAX_LEN)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=0)

    # 5. Get Probabilities from BERT Model
    all_cve_probs = get_predictions(model, loader)

    # 6. Define evaluation space based on BERT's known classes
    bert_parent_classes = sorted(list(set([parent_map.get(t, t) for t in bert_classes])))
    num_classes = len(bert_parent_classes)
    logging.info(f"Total Parent Classes handled by BERT model: {num_classes}")

    # 7. Evaluate Loop
    bert_metrics = {'R@1': [], 'R@5': [], 'R@10': []}
    y_true_all = []
    y_score_all = []
    valid_cves = 0

    logging.info("Calculating Advanced Metrics...")
    for idx, cve_desc in enumerate(cve_texts):
        ground_truth = set(cve_ground_truths[idx])
        n_true = len(ground_truth)
        
        if n_true == 0:
            continue
            
        valid_cves += 1
        probs = all_cve_probs[idx]

        # Roll up sub-technique probabilities to parent techniques
        parent_probs = {tid: 0.0 for tid in bert_parent_classes}
        for i, class_label in enumerate(bert_classes):
            parent_tc = parent_map.get(class_label, class_label)
            if parent_tc in parent_probs:
                # If multiple sub-techniques share a parent, keep the highest probability
                parent_probs[parent_tc] = max(parent_probs[parent_tc], probs[i])

        # Populate Metric Matrices
        sample_y_true = np.zeros(num_classes)
        sample_y_score = np.zeros(num_classes)
        
        for i, tid in enumerate(bert_parent_classes):
            sample_y_score[i] = parent_probs[tid]
            if tid in ground_truth:
                sample_y_true[i] = 1.0
                
        y_true_all.append(sample_y_true)
        y_score_all.append(sample_y_score)
        
        # Calculate Recall @ K
        sorted_indices = np.argsort(sample_y_score)[::-1]
        sorted_tcodes = [bert_parent_classes[i] for i in sorted_indices]
                
        for k in [1, 5, 10]:
            s_hits = len(set(sorted_tcodes[:k]).intersection(ground_truth))
            bert_metrics[f'R@{k}'].append(s_hits / n_true)

    # Convert arrays for sklearn
    y_true_all = np.array(y_true_all)
    y_score_all = np.array(y_score_all)

    # Calculate Advanced Metrics
    lrap = label_ranking_average_precision_score(y_true_all, y_score_all)
    rl = label_ranking_loss(y_true_all, y_score_all)
    ce = coverage_error(y_true_all, y_score_all)

    # Print Final Averaged Results
    print("\n" + "="*50)
    print(f"ATTACK-BERT EVALUATION RESULTS (Tested on {valid_cves} CVEs)")
    print("="*50)
    print(f"Recall@1:       {(np.mean(bert_metrics['R@1']) * 100):.2f}%")
    print(f"Recall@5:       {(np.mean(bert_metrics['R@5']) * 100):.2f}%")
    print(f"Recall@10:      {(np.mean(bert_metrics['R@10']) * 100):.2f}%")
    print("-" * 50)
    print(f"LRAP:           {lrap * 100:.2f}%  (Higher is better)")
    print(f"Ranking Loss:   {rl:.4f}   (Lower is better)")
    print(f"Coverage Error: {ce:.4f}   (Lower is better)")
    print("="*50)

if __name__ == "__main__":
    main()

2026-04-05 03:51:43,725 [INFO] Loading Fine-Tuned BERT model and Tokenizer...
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13171.91it/s]
2026-04-05 03:51:44,059 [INFO] Model loaded. Output layer size: 105
2026-04-05 03:51:44,059 [INFO] Loading ATT&CK STIX data & Benchmark Data...
2026-04-05 03:51:44,679 [INFO] Running Inference on CVE Benchmark Dataset...
BERT Batches: 100%|██████████| 10/10 [00:01<00:00,  8.10it/s]
2026-04-05 03:51:45,916 [INFO] Total Parent Classes handled by BERT model: 105
2026-04-05 03:51:45,916 [INFO] Calculating Advanced Metrics...



ATTACK-BERT EVALUATION RESULTS (Tested on 302 CVEs)
Recall@1:       7.56%
Recall@5:       14.96%
Recall@10:      23.81%
--------------------------------------------------
LRAP:           17.33%  (Higher is better)
Ranking Loss:   0.5207   (Lower is better)
Coverage Error: 63.4338   (Lower is better)


# DistilRoberta Exp

In [6]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
import logging
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.utils.tensorboard import SummaryWriter
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, label_ranking_average_precision_score

# --- Configuration ---
DATA_DIR = "./processed_data_parent"
MODEL_DIR = "./models/transformer_parent" # New directory for the better model
LOG_DIR = "./runs/transformer_experiment_parent"
MODEL_NAME = "distilroberta-base"            # The Cybersecurity Specialist          
MAX_LEN = 256
BATCH_SIZE = 64   
EPOCHS = 10
LEARNING_RATE = 2e-5
LOG_INTERVAL = 1  

# Setup Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
writer = SummaryWriter(LOG_DIR)

if torch.cuda.is_available():
    device = torch.device('cuda')
    logging.info(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    logging.warning("GPU not detected! Training will be slow.")

class CVEDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',     
            truncation=True,          
            return_attention_mask=True,
            return_tensors='pt',      
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.FloatTensor(label)
        }

def load_data():
    logging.info("Loading Data...")
    train_text = pd.read_csv(os.path.join(DATA_DIR, "train_text.csv"))['text'].fillna("").values
    test_text = pd.read_csv(os.path.join(DATA_DIR, "test_text.csv"))['text'].fillna("").values
    y_train = np.load(os.path.join(DATA_DIR, "y_train.npz"))['arr_0']
    y_test = np.load(os.path.join(DATA_DIR, "y_test.npz"))['arr_0']
    
    with open(os.path.join(DATA_DIR, "mlb.pkl"), "rb") as f:
        mlb = pickle.load(f)
        
    return train_text, test_text, y_train, y_test, len(mlb.classes_)

def train_epoch(model, data_loader, loss_fn, optimizer, scheduler, device, n_examples):
    model = model.train()
    losses = []
    
    total_steps = len(data_loader)
    for i, d in enumerate(data_loader):
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        targets = d["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        logits = outputs.logits
        loss = loss_fn(logits, targets)

        losses.append(loss.item())
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        if (i + 1) % LOG_INTERVAL == 0:
            avg_loss = np.mean(losses[-LOG_INTERVAL:])
            global_step = (n_examples * total_steps) + i
            logging.info(f"Epoch [{n_examples+1}/{EPOCHS}] Step [{i+1}/{total_steps}] Loss: {avg_loss:.4f}")
            writer.add_scalar('Training Loss', avg_loss, global_step)

    return np.mean(losses)

def eval_model(model, data_loader, loss_fn, device):
    model = model.eval()
    losses = []
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            loss = loss_fn(outputs.logits, targets)
            losses.append(loss.item())
            
            probs = torch.sigmoid(outputs.logits).cpu().numpy()
            targets = targets.cpu().numpy()
            
            all_preds.extend(probs)
            all_targets.extend(targets)

    return np.mean(losses), np.array(all_preds), np.array(all_targets)

def main():
    X_train, X_test, y_train, y_test, n_classes = load_data()
    
    logging.info(f"Loading Tokenizer ({MODEL_NAME})...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    train_ds = CVEDataset(X_train, y_train, tokenizer, MAX_LEN)
    test_ds = CVEDataset(X_test, y_test, tokenizer, MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=0)
    
    logging.info(f"Initializing Model (Classes: {n_classes})...")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, 
        num_labels=n_classes,
        problem_type="multi_label_classification"
    )
    model = model.to(device)
    
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    
    loss_fn = torch.nn.BCEWithLogitsLoss()
    
    logging.info("Starting Training with ATTACK-BERT...")
    
    # Track the BEST LRAP instead of F1
    best_lrap = 0.0 
    
    if not os.path.exists(MODEL_DIR):
        os.makedirs(MODEL_DIR)

    for epoch in range(EPOCHS):
        logging.info(f"Epoch {epoch + 1}/{EPOCHS}")
        
        train_loss = train_epoch(
            model, train_loader, loss_fn, optimizer, scheduler, device, epoch
        )
        
        logging.info(f"Train loss: {train_loss:.4f}")
        
        val_loss, preds, targets = eval_model(
            model, test_loader, loss_fn, device
        )
        
        # Calculate standard F1 just for logging
        preds_binary = (preds > 0.5).astype(int)
        val_f1 = f1_score(targets, preds_binary, average='micro')
        
        # Calculate LRAP for actual model selection!
        val_lrap = label_ranking_average_precision_score(targets, preds)
        
        logging.info(f"Val loss: {val_loss:.4f} | Micro F1: {val_f1:.4f} | LRAP: {val_lrap:.4f}")
        writer.add_scalar('Validation Loss', val_loss, epoch)
        writer.add_scalar('Validation F1', val_f1, epoch)     
        writer.add_scalar('Validation LRAP', val_lrap, epoch)

        # Save model based on LRAP
        if val_lrap > best_lrap:
            logging.info(f"New Best LRAP! ({val_lrap:.4f} > {best_lrap:.4f}). Saving model...")            
            model.save_pretrained(MODEL_DIR)
            tokenizer.save_pretrained(MODEL_DIR)
            best_lrap = val_lrap

    logging.info(f"Training Complete. Best Validation LRAP: {best_lrap:.4f}")
    writer.close()

if __name__ == "__main__":
    main()

2026-04-05 03:51:46,060 [INFO] Using GPU: NVIDIA GeForce RTX 3090
2026-04-05 03:51:46,063 [INFO] Loading Data...
2026-04-05 03:51:46,326 [INFO] Loading Tokenizer (distilroberta-base)...
2026-04-05 03:51:46,397 [INFO] HTTP Request: HEAD https://huggingface.co/distilroberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
2026-04-05 03:51:46,443 [INFO] HTTP Request: HEAD https://huggingface.co/distilroberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-05 03:51:46,476 [INFO] HTTP Request: GET https://huggingface.co/api/models/distilroberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-05 03:51:46,510 [INFO] HTTP Request: GET https://huggingface.co/api/models/distilbert/distilroberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-05 03:51:46,560 [INFO] HTTP Request: GET https://huggingface.co/api/models/distilroberta-base/tree/main?recursive=true&exp

In [7]:
import torch
import numpy as np
import pandas as pd
import pickle
import logging
import os
import json
import ast
import re
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    coverage_error, 
    label_ranking_loss, 
    label_ranking_average_precision_score
)

# --- Configuration ---
DATA_DIR = "./processed_data_parent"
MODEL_DIR = "./models/transformer_parent"
BATCH_SIZE = 32
MAX_LEN = 256

# Setup Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
    logging.warning("Using CPU. Inference might be slow.")

class EvaluationDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

def get_predictions(model, loader):
    model.eval()
    all_probs = []
    
    logging.info("Running Inference on CVE Benchmark Dataset...")
    with torch.no_grad():
        for d in tqdm(loader, desc="BERT Batches"):
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits)
            all_probs.extend(probs.cpu().numpy())
            
    return np.array(all_probs)

def main():
    # 1. Load the BERT Model and Class Names (MLB)
    logging.info("Loading Fine-Tuned BERT model and Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
    model.to(device)
    
    with open(os.path.join(DATA_DIR, "mlb.pkl"), "rb") as f:
        mlb = pickle.load(f)
        bert_classes = mlb.classes_
        
    logging.info(f"Model loaded. Output layer size: {len(bert_classes)}")

    # 2. Load Mappings (STIX & id2mitre)
    logging.info("Loading ATT&CK STIX data & Benchmark Data...")
    try:
        id2mitre = json.load(open('id2mitre.json', 'r'))
        with open(r"OSRs\ATTACK\enterprise-attack-v16.1.json", "r", encoding="utf-8") as f:
            stix_bundle = json.load(f)
    except Exception as e:
        logging.error(f"Could not load JSON mapping files: {e}")
        return

    name_to_tcode = {}
    parent_map = {}

    for obj in stix_bundle.get("objects", []):
        if obj.get("type") == "attack-pattern" and not obj.get("revoked") and not obj.get("x_mitre_deprecated"):
            t_code = next((ref.get("external_id") for ref in obj.get("external_references", []) if ref.get("source_name") == "mitre-attack"), None)
            if t_code:
                name_to_tcode[obj.get("name")] = t_code

    for obj in stix_bundle.get("objects", []):
        if obj.get("type") == "relationship" and obj.get("relationship_type") == "subtechnique-of":
            sub_id, parent_id = obj.get("source_ref"), obj.get("target_ref")
            sub_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == sub_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
            parent_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == parent_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
            if sub_tcode and parent_tcode:
                parent_map[sub_tcode] = parent_tcode

    # 3. Load CVE Dataset
    smet_raw = pd.read_excel("CVE_annotated_dataset.xlsx")
    smet_records = []
    for _, row in smet_raw.iterrows():
        try:
            tech_names = ast.literal_eval(row["ATT&CK Techniques"])
        except:
            tech_names = []
            
        t_codes = []
        for name in tech_names:
            mapped_id = next((k for k, v in id2mitre.items() if name in v), name_to_tcode.get(name))
            if mapped_id:
                t_codes.append(parent_map.get(mapped_id, mapped_id))
                
        if t_codes:
            smet_records.append({
                "Description": row["Description"],
                "T_Codes": list(set(t_codes))
            })
            
    smet_df = pd.DataFrame(smet_records)
    cve_texts = smet_df['Description'].tolist()
    cve_ground_truths = smet_df['T_Codes'].tolist()

    # 4. Prepare Data Loader for CVE texts
    dataset = EvaluationDataset(cve_texts, tokenizer, MAX_LEN)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=0)

    # 5. Get Probabilities from BERT Model
    all_cve_probs = get_predictions(model, loader)

    # 6. Define evaluation space based on BERT's known classes
    bert_parent_classes = sorted(list(set([parent_map.get(t, t) for t in bert_classes])))
    num_classes = len(bert_parent_classes)
    logging.info(f"Total Parent Classes handled by BERT model: {num_classes}")

    # 7. Evaluate Loop
    bert_metrics = {'R@1': [], 'R@5': [], 'R@10': []}
    y_true_all = []
    y_score_all = []
    valid_cves = 0

    logging.info("Calculating Advanced Metrics...")
    for idx, cve_desc in enumerate(cve_texts):
        ground_truth = set(cve_ground_truths[idx])
        n_true = len(ground_truth)
        
        if n_true == 0:
            continue
            
        valid_cves += 1
        probs = all_cve_probs[idx]

        # Roll up sub-technique probabilities to parent techniques
        parent_probs = {tid: 0.0 for tid in bert_parent_classes}
        for i, class_label in enumerate(bert_classes):
            parent_tc = parent_map.get(class_label, class_label)
            if parent_tc in parent_probs:
                # If multiple sub-techniques share a parent, keep the highest probability
                parent_probs[parent_tc] = max(parent_probs[parent_tc], probs[i])

        # Populate Metric Matrices
        sample_y_true = np.zeros(num_classes)
        sample_y_score = np.zeros(num_classes)
        
        for i, tid in enumerate(bert_parent_classes):
            sample_y_score[i] = parent_probs[tid]
            if tid in ground_truth:
                sample_y_true[i] = 1.0
                
        y_true_all.append(sample_y_true)
        y_score_all.append(sample_y_score)
        
        # Calculate Recall @ K
        sorted_indices = np.argsort(sample_y_score)[::-1]
        sorted_tcodes = [bert_parent_classes[i] for i in sorted_indices]
                
        for k in [1, 5, 10]:
            s_hits = len(set(sorted_tcodes[:k]).intersection(ground_truth))
            bert_metrics[f'R@{k}'].append(s_hits / n_true)

    # Convert arrays for sklearn
    y_true_all = np.array(y_true_all)
    y_score_all = np.array(y_score_all)

    # Calculate Advanced Metrics
    lrap = label_ranking_average_precision_score(y_true_all, y_score_all)
    rl = label_ranking_loss(y_true_all, y_score_all)
    ce = coverage_error(y_true_all, y_score_all)

    # Print Final Averaged Results
    print("\n" + "="*50)
    print(f"ATTACK-BERT EVALUATION RESULTS (Tested on {valid_cves} CVEs)")
    print("="*50)
    print(f"Recall@1:       {(np.mean(bert_metrics['R@1']) * 100):.2f}%")
    print(f"Recall@5:       {(np.mean(bert_metrics['R@5']) * 100):.2f}%")
    print(f"Recall@10:      {(np.mean(bert_metrics['R@10']) * 100):.2f}%")
    print("-" * 50)
    print(f"LRAP:           {lrap * 100:.2f}%  (Higher is better)")
    print(f"Ranking Loss:   {rl:.4f}   (Lower is better)")
    print(f"Coverage Error: {ce:.4f}   (Lower is better)")
    print("="*50)

if __name__ == "__main__":
    main()

2026-04-05 04:40:56,900 [INFO] Loading Fine-Tuned BERT model and Tokenizer...
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6999.62it/s]
2026-04-05 04:40:57,640 [INFO] Model loaded. Output layer size: 105
2026-04-05 04:40:57,641 [INFO] Loading ATT&CK STIX data & Benchmark Data...
2026-04-05 04:40:57,834 [INFO] Running Inference on CVE Benchmark Dataset...
BERT Batches: 100%|██████████| 10/10 [00:00<00:00, 14.58it/s]
2026-04-05 04:40:58,524 [INFO] Total Parent Classes handled by BERT model: 105
2026-04-05 04:40:58,524 [INFO] Calculating Advanced Metrics...



ATTACK-BERT EVALUATION RESULTS (Tested on 302 CVEs)
Recall@1:       8.94%
Recall@5:       13.63%
Recall@10:      21.74%
--------------------------------------------------
LRAP:           17.88%  (Higher is better)
Ranking Loss:   0.5311   (Lower is better)
Coverage Error: 64.7417   (Lower is better)


# SecureBert+ Exp

In [8]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
import logging
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.utils.tensorboard import SummaryWriter
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, label_ranking_average_precision_score

# --- Configuration ---
DATA_DIR = "./processed_data_parent"
MODEL_DIR = "./models/SecureBERT_finetuned_parent" # New directory for the better model
LOG_DIR = "./runs/SecureBERT_experiment_parent"
MODEL_NAME = "ehsanaghaei/SecureBERT_Plus"            # The Cybersecurity Specialist         
MAX_LEN = 256
BATCH_SIZE = 64   
EPOCHS = 10
LEARNING_RATE = 2e-5
LOG_INTERVAL = 1  

# Setup Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
writer = SummaryWriter(LOG_DIR)

if torch.cuda.is_available():
    device = torch.device('cuda')
    logging.info(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    logging.warning("GPU not detected! Training will be slow.")

class CVEDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',     
            truncation=True,          
            return_attention_mask=True,
            return_tensors='pt',      
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.FloatTensor(label)
        }

def load_data():
    logging.info("Loading Data...")
    train_text = pd.read_csv(os.path.join(DATA_DIR, "train_text.csv"))['text'].fillna("").values
    test_text = pd.read_csv(os.path.join(DATA_DIR, "test_text.csv"))['text'].fillna("").values
    y_train = np.load(os.path.join(DATA_DIR, "y_train.npz"))['arr_0']
    y_test = np.load(os.path.join(DATA_DIR, "y_test.npz"))['arr_0']
    
    with open(os.path.join(DATA_DIR, "mlb.pkl"), "rb") as f:
        mlb = pickle.load(f)
        
    return train_text, test_text, y_train, y_test, len(mlb.classes_)

def train_epoch(model, data_loader, loss_fn, optimizer, scheduler, device, n_examples):
    model = model.train()
    losses = []
    
    total_steps = len(data_loader)
    for i, d in enumerate(data_loader):
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        targets = d["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        logits = outputs.logits
        loss = loss_fn(logits, targets)

        losses.append(loss.item())
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        if (i + 1) % LOG_INTERVAL == 0:
            avg_loss = np.mean(losses[-LOG_INTERVAL:])
            global_step = (n_examples * total_steps) + i
            logging.info(f"Epoch [{n_examples+1}/{EPOCHS}] Step [{i+1}/{total_steps}] Loss: {avg_loss:.4f}")
            writer.add_scalar('Training Loss', avg_loss, global_step)

    return np.mean(losses)

def eval_model(model, data_loader, loss_fn, device):
    model = model.eval()
    losses = []
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            loss = loss_fn(outputs.logits, targets)
            losses.append(loss.item())
            
            probs = torch.sigmoid(outputs.logits).cpu().numpy()
            targets = targets.cpu().numpy()
            
            all_preds.extend(probs)
            all_targets.extend(targets)

    return np.mean(losses), np.array(all_preds), np.array(all_targets)

def main():
    X_train, X_test, y_train, y_test, n_classes = load_data()
    
    logging.info(f"Loading Tokenizer ({MODEL_NAME})...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    train_ds = CVEDataset(X_train, y_train, tokenizer, MAX_LEN)
    test_ds = CVEDataset(X_test, y_test, tokenizer, MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=0)
    
    logging.info(f"Initializing Model (Classes: {n_classes})...")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, 
        num_labels=n_classes,
        problem_type="multi_label_classification"
    )
    model = model.to(device)
    
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    
    loss_fn = torch.nn.BCEWithLogitsLoss()
    
    logging.info("Starting Training with ATTACK-BERT...")
    
    # Track the BEST LRAP instead of F1
    best_lrap = 0.0 
    
    if not os.path.exists(MODEL_DIR):
        os.makedirs(MODEL_DIR)

    for epoch in range(EPOCHS):
        logging.info(f"Epoch {epoch + 1}/{EPOCHS}")
        
        train_loss = train_epoch(
            model, train_loader, loss_fn, optimizer, scheduler, device, epoch
        )
        
        logging.info(f"Train loss: {train_loss:.4f}")
        
        val_loss, preds, targets = eval_model(
            model, test_loader, loss_fn, device
        )
        
        # Calculate standard F1 just for logging
        preds_binary = (preds > 0.5).astype(int)
        val_f1 = f1_score(targets, preds_binary, average='micro')
        
        # Calculate LRAP for actual model selection!
        val_lrap = label_ranking_average_precision_score(targets, preds)
        
        logging.info(f"Val loss: {val_loss:.4f} | Micro F1: {val_f1:.4f} | LRAP: {val_lrap:.4f}")
        writer.add_scalar('Validation Loss', val_loss, epoch)
        writer.add_scalar('Validation F1', val_f1, epoch)     
        writer.add_scalar('Validation LRAP', val_lrap, epoch)

        # Save model based on LRAP
        if val_lrap > best_lrap:
            logging.info(f"New Best LRAP! ({val_lrap:.4f} > {best_lrap:.4f}). Saving model...")            
            model.save_pretrained(MODEL_DIR)
            tokenizer.save_pretrained(MODEL_DIR)
            best_lrap = val_lrap

    logging.info(f"Training Complete. Best Validation LRAP: {best_lrap:.4f}")
    writer.close()

if __name__ == "__main__":
    main()

2026-04-05 04:40:58,612 [INFO] Using GPU: NVIDIA GeForce RTX 3090
2026-04-05 04:40:58,615 [INFO] Loading Data...
2026-04-05 04:40:58,944 [INFO] Loading Tokenizer (ehsanaghaei/SecureBERT_Plus)...
2026-04-05 04:40:59,008 [INFO] HTTP Request: HEAD https://huggingface.co/ehsanaghaei/SecureBERT_Plus/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-05 04:40:59,019 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ehsanaghaei/SecureBERT_Plus/85a7ab3c9013ef77d38a73b52e5a87293301d543/config.json "HTTP/1.1 200 OK"
2026-04-05 04:40:59,058 [INFO] HTTP Request: HEAD https://huggingface.co/ehsanaghaei/SecureBERT_Plus/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-05 04:40:59,071 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ehsanaghaei/SecureBERT_Plus/85a7ab3c9013ef77d38a73b52e5a87293301d543/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-05 04:40:59,116 [INFO] HTTP Request: GET https://huggingface.co

In [9]:
import torch
import numpy as np
import pandas as pd
import pickle
import logging
import os
import json
import ast
import re
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    coverage_error, 
    label_ranking_loss, 
    label_ranking_average_precision_score
)

# --- Configuration ---
DATA_DIR = "./processed_data_parent"
MODEL_DIR = "./models/SecureBERT_finetuned_parent"
BATCH_SIZE = 32
MAX_LEN = 256

# Setup Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
    logging.warning("Using CPU. Inference might be slow.")

class EvaluationDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

def get_predictions(model, loader):
    model.eval()
    all_probs = []
    
    logging.info("Running Inference on CVE Benchmark Dataset...")
    with torch.no_grad():
        for d in tqdm(loader, desc="BERT Batches"):
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits)
            all_probs.extend(probs.cpu().numpy())
            
    return np.array(all_probs)

def main():
    # 1. Load the BERT Model and Class Names (MLB)
    logging.info("Loading Fine-Tuned BERT model and Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
    model.to(device)
    
    with open(os.path.join(DATA_DIR, "mlb.pkl"), "rb") as f:
        mlb = pickle.load(f)
        bert_classes = mlb.classes_
        
    logging.info(f"Model loaded. Output layer size: {len(bert_classes)}")

    # 2. Load Mappings (STIX & id2mitre)
    logging.info("Loading ATT&CK STIX data & Benchmark Data...")
    try:
        id2mitre = json.load(open('id2mitre.json', 'r'))
        with open(r"OSRs\ATTACK\enterprise-attack-v16.1.json", "r", encoding="utf-8") as f:
            stix_bundle = json.load(f)
    except Exception as e:
        logging.error(f"Could not load JSON mapping files: {e}")
        return

    name_to_tcode = {}
    parent_map = {}

    for obj in stix_bundle.get("objects", []):
        if obj.get("type") == "attack-pattern" and not obj.get("revoked") and not obj.get("x_mitre_deprecated"):
            t_code = next((ref.get("external_id") for ref in obj.get("external_references", []) if ref.get("source_name") == "mitre-attack"), None)
            if t_code:
                name_to_tcode[obj.get("name")] = t_code

    for obj in stix_bundle.get("objects", []):
        if obj.get("type") == "relationship" and obj.get("relationship_type") == "subtechnique-of":
            sub_id, parent_id = obj.get("source_ref"), obj.get("target_ref")
            sub_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == sub_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
            parent_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == parent_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
            if sub_tcode and parent_tcode:
                parent_map[sub_tcode] = parent_tcode

    # 3. Load CVE Dataset
    smet_raw = pd.read_excel("CVE_annotated_dataset.xlsx")
    smet_records = []
    for _, row in smet_raw.iterrows():
        try:
            tech_names = ast.literal_eval(row["ATT&CK Techniques"])
        except:
            tech_names = []
            
        t_codes = []
        for name in tech_names:
            mapped_id = next((k for k, v in id2mitre.items() if name in v), name_to_tcode.get(name))
            if mapped_id:
                t_codes.append(parent_map.get(mapped_id, mapped_id))
                
        if t_codes:
            smet_records.append({
                "Description": row["Description"],
                "T_Codes": list(set(t_codes))
            })
            
    smet_df = pd.DataFrame(smet_records)
    cve_texts = smet_df['Description'].tolist()
    cve_ground_truths = smet_df['T_Codes'].tolist()

    # 4. Prepare Data Loader for CVE texts
    dataset = EvaluationDataset(cve_texts, tokenizer, MAX_LEN)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=0)

    # 5. Get Probabilities from BERT Model
    all_cve_probs = get_predictions(model, loader)

    # 6. Define evaluation space based on BERT's known classes
    bert_parent_classes = sorted(list(set([parent_map.get(t, t) for t in bert_classes])))
    num_classes = len(bert_parent_classes)
    logging.info(f"Total Parent Classes handled by BERT model: {num_classes}")

    # 7. Evaluate Loop
    bert_metrics = {'R@1': [], 'R@5': [], 'R@10': []}
    y_true_all = []
    y_score_all = []
    valid_cves = 0

    logging.info("Calculating Advanced Metrics...")
    for idx, cve_desc in enumerate(cve_texts):
        ground_truth = set(cve_ground_truths[idx])
        n_true = len(ground_truth)
        
        if n_true == 0:
            continue
            
        valid_cves += 1
        probs = all_cve_probs[idx]

        # Roll up sub-technique probabilities to parent techniques
        parent_probs = {tid: 0.0 for tid in bert_parent_classes}
        for i, class_label in enumerate(bert_classes):
            parent_tc = parent_map.get(class_label, class_label)
            if parent_tc in parent_probs:
                # If multiple sub-techniques share a parent, keep the highest probability
                parent_probs[parent_tc] = max(parent_probs[parent_tc], probs[i])

        # Populate Metric Matrices
        sample_y_true = np.zeros(num_classes)
        sample_y_score = np.zeros(num_classes)
        
        for i, tid in enumerate(bert_parent_classes):
            sample_y_score[i] = parent_probs[tid]
            if tid in ground_truth:
                sample_y_true[i] = 1.0
                
        y_true_all.append(sample_y_true)
        y_score_all.append(sample_y_score)
        
        # Calculate Recall @ K
        sorted_indices = np.argsort(sample_y_score)[::-1]
        sorted_tcodes = [bert_parent_classes[i] for i in sorted_indices]
                
        for k in [1, 5, 10]:
            s_hits = len(set(sorted_tcodes[:k]).intersection(ground_truth))
            bert_metrics[f'R@{k}'].append(s_hits / n_true)

    # Convert arrays for sklearn
    y_true_all = np.array(y_true_all)
    y_score_all = np.array(y_score_all)

    # Calculate Advanced Metrics
    lrap = label_ranking_average_precision_score(y_true_all, y_score_all)
    rl = label_ranking_loss(y_true_all, y_score_all)
    ce = coverage_error(y_true_all, y_score_all)

    # Print Final Averaged Results
    print("\n" + "="*50)
    print(f"ATTACK-BERT EVALUATION RESULTS (Tested on {valid_cves} CVEs)")
    print("="*50)
    print(f"Recall@1:       {(np.mean(bert_metrics['R@1']) * 100):.2f}%")
    print(f"Recall@5:       {(np.mean(bert_metrics['R@5']) * 100):.2f}%")
    print(f"Recall@10:      {(np.mean(bert_metrics['R@10']) * 100):.2f}%")
    print("-" * 50)
    print(f"LRAP:           {lrap * 100:.2f}%  (Higher is better)")
    print(f"Ranking Loss:   {rl:.4f}   (Lower is better)")
    print(f"Coverage Error: {ce:.4f}   (Lower is better)")
    print("="*50)

if __name__ == "__main__":
    main()

2026-04-05 06:16:45,584 [INFO] Loading Fine-Tuned BERT model and Tokenizer...
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7417.54it/s]
2026-04-05 06:16:46,281 [INFO] Model loaded. Output layer size: 105
2026-04-05 06:16:46,282 [INFO] Loading ATT&CK STIX data & Benchmark Data...
2026-04-05 06:16:46,455 [INFO] Running Inference on CVE Benchmark Dataset...
BERT Batches: 100%|██████████| 10/10 [00:01<00:00,  8.70it/s]
2026-04-05 06:16:47,606 [INFO] Total Parent Classes handled by BERT model: 105
2026-04-05 06:16:47,606 [INFO] Calculating Advanced Metrics...



ATTACK-BERT EVALUATION RESULTS (Tested on 302 CVEs)
Recall@1:       6.29%
Recall@5:       11.92%
Recall@10:      21.74%
--------------------------------------------------
LRAP:           16.30%  (Higher is better)
Ranking Loss:   0.5207   (Lower is better)
Coverage Error: 63.3543   (Lower is better)


# LLM Experiments

In [ ]:
# Cell 1 — Setup: load ATT&CK catalogue + LM Studio connectivity check

import json, re, requests

# ── 1. Load ATT&CK STIX ───────────────────────────────────────────
ATTACK_JSON = "OSRs/ATTACK/enterprise-attack-v16.1.json"
with open(ATTACK_JSON, encoding="utf-8") as f:
    stix_bundle = json.load(f)

objects = stix_bundle.get("objects", [])

# ── 2. Build technique catalogue ──────────────────────────────────
parent_map  = {}   # sub-tcode → parent tcode
revoked_map = {}   # old tcode → new tcode

def get_tcode(obj):
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id", "")
    return ""

# Index all objects by id
stix_by_id = {o["id"]: o for o in objects}

# Build maps
for o in objects:
    if o.get("type") != "relationship":
        continue
    src = stix_by_id.get(o.get("source_ref",""))
    tgt = stix_by_id.get(o.get("target_ref",""))
    if not src or not tgt:
        continue
    tc_src = get_tcode(src)
    tc_tgt = get_tcode(tgt)
    rt = o.get("relationship_type","")
    if rt == "subtechnique-of" and tc_src and tc_tgt:
        parent_map[tc_src] = tc_tgt
    elif rt == "revoked-by" and tc_src and tc_tgt:
        revoked_map[tc_src] = tc_tgt

def resolve_to_parent(tc):
    tc = revoked_map.get(tc, tc)
    tc = parent_map.get(tc, tc)
    tc = revoked_map.get(tc, tc)
    return tc

# Active parent techniques only
revoked_tcodes = set(revoked_map.keys())
for o in objects:
    if o.get("type") == "attack-pattern":
        if o.get("x_mitre_revoked") or o.get("revoked"):
            tc = get_tcode(o)
            if tc:
                revoked_tcodes.add(tc)

technique_ids   = []   # list of T-codes
technique_names = {}   # T-code → name
technique_descs = {}   # T-code → description

for o in objects:
    if o.get("type") != "attack-pattern":
        continue
    if o.get("x_mitre_revoked") or o.get("revoked"):
        continue
    tc = get_tcode(o)
    if not tc or "." in tc:   # skip sub-techniques
        continue
    if tc in revoked_tcodes:
        continue
    technique_ids.append(tc)
    technique_names[tc] = o.get("name", "")
    technique_descs[tc] = o.get("description", "")

technique_ids = sorted(set(technique_ids))
technique_id_set = set(technique_ids)

print(f"Active parent techniques: {len(technique_ids)}")
print(f"Sample: {technique_ids[:5]}")

# ── 3. Build technique list string for prompt ─────────────────────
technique_list_str = "\n".join(
    f"{tc}: {technique_names[tc]}"
    for tc in technique_ids
)
print(f"\nTechnique catalogue sample:")
print("\n".join(technique_list_str.split("\n")[:5]))
print("...")

# ── 4. LM Studio config ───────────────────────────────────────────
LM_STUDIO_URL = "http://localhost:1234/v1/chat/completions"
LM_MODEL      = "google/gemma-4-26b-a4b"   # label for logging only
TOP_K         = 5

SYSTEM_PROMPT = """You are a cybersecurity expert specializing in MITRE ATT&CK threat intelligence.

Your task: given a CVE vulnerability description, identify the most relevant ATT&CK techniques an adversary would use to exploit it.

Rules:
- You MUST only select techniques from the provided catalogue. Do not invent T-codes.
- Return EXACTLY a JSON object with one key "techniques" containing a list of T-codes, ranked by relevance (most relevant first).
- Return NO other text, explanation, or markdown. Only the raw JSON object.
- Select between 1 and 5 techniques. Only include techniques that are clearly relevant.

Output format (strictly follow this):
{"techniques": ["T1190", "T1059", "T1078"]}"""

def build_user_prompt(cve_description, db_hint=None):
    prompt = (f"CVE Description:\n{cve_description}\n\n"
              f"Valid ATT&CK Technique Catalogue (T-code: Name):\n"
              f"{technique_list_str}\n\n"
              f"Identify the top-{TOP_K} most relevant ATT&CK parent "
              f"techniques for this CVE.")
    if db_hint:
        prompt += (f"\n\nHint: A vulnerability database suggests these "
                   f"techniques may be relevant: {db_hint}\n"
                   f"Use this as a guide but apply your own judgment.")
    return prompt

def call_llm(cve_description, db_hint=None, retries=2):
    payload = {
        "model"      : LM_MODEL,
        "messages"   : [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_prompt(
                                    cve_description, db_hint)},
        ],
        "temperature": 0.0,
        "max_tokens" : 4096,
        "stream"     : False,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    for attempt in range(retries + 1):
        try:
            resp    = requests.post(LM_STUDIO_URL, json=payload, timeout=90)
            resp.raise_for_status()
            content = resp.json()["choices"][0]["message"]["content"].strip()
            content = re.sub(r'^```json\s*|^```\s*|```$', '',
                             content, flags=re.MULTILINE).strip()
            parsed  = json.loads(content)
            raw_tcs = parsed.get("techniques", [])
            valid   = [tc.strip().upper() for tc in raw_tcs
                       if tc.strip().upper() in technique_id_set]
            return valid, content
        except requests.exceptions.ConnectionError:
            print("✗ Cannot connect — is LM Studio server running on port 1234?")
            return [], ""
        except json.JSONDecodeError:
            if attempt < retries:
                import time; time.sleep(1)
                continue
            found = re.findall(r'T\d{4}', content)
            valid = [tc.upper() for tc in found
                     if tc.upper() in technique_id_set]
            return valid[:TOP_K], content
        except Exception as e:
            print(f"  Error: {e}")
            return [], ""

# ── 5. Connectivity check ─────────────────────────────────────────
print("\nTesting LM Studio connection...")
test_tcs, test_raw = call_llm(
    "SQL injection vulnerability allows unauthenticated remote "
    "attackers to execute arbitrary SQL commands via user input.")
if test_tcs:
    print(f"✓ Connected.")
    print(f"  Predicted : {test_tcs}")
    print(f"  Raw output: {test_raw}")
else:
    print(f"✗ Failed. Raw: {test_raw}")
    print("  → Start LM Studio → Local Server → Load model → Start server")

Active parent techniques: 214
Sample: ['T1001', 'T1003', 'T1005', 'T1006', 'T1007']

Technique catalogue sample:
T1001: Data Obfuscation
T1003: OS Credential Dumping
T1005: Data from Local System
T1006: Direct Volume Access
T1007: System Service Discovery
...

Testing LM Studio connection...
✓ Connected.
  Predicted : ['T1190', 'T1210', 'T1059', 'T1202', 'T1659']
  Raw output: {"techniques": ["T1190", "T1210", "T1059", "T1202", "T1659"]}


In [ ]:
# Cell 2 — Full KEV + SMET Evaluation with Gemma-4

import ast, time, glob
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path

LM_MODEL = "gemma-4-26b-a4b"   # update to exact name shown in LM Studio

# ══════════════════════════════════════════════════════════════════
# A. Load eval datasets
# ══════════════════════════════════════════════════════════════════

# ── KEV ───────────────────────────────────────────────────────────
with open("OSRs/kev-07.28.2025_attack-16.1-enterprise.json",
          encoding="utf-8") as f:
    kev_data = json.load(f)

kev_by_cve = {}
for entry in kev_data["mapping_objects"]:
    cve_id = entry.get("capability_id","").strip()
    tc_raw = entry.get("attack_object_id","").strip()
    desc   = entry.get("capability_description","").strip()
    if not cve_id.startswith("CVE-") or not tc_raw:
        continue
    parent_tc = parent_map.get(tc_raw, tc_raw)
    if "." in parent_tc or parent_tc not in technique_id_set:
        continue
    if cve_id not in kev_by_cve:
        kev_by_cve[cve_id] = {"desc": desc, "techs": set()}
    kev_by_cve[cve_id]["techs"].add(parent_tc)

# Enrich KEV with NVD descriptions
print("Loading NVD for KEV enrichment...")
nvd_lookup = {}
for fpath in sorted(glob.glob("OSRs/NVD/nvdcve-2.0-*.json")):
    with open(fpath, encoding="utf-8") as f:
        data = json.load(f)
    for entry in data.get("vulnerabilities", []):
        cve_obj = entry.get("cve", {})
        cve_id  = cve_obj.get("id", "")
        for d in cve_obj.get("descriptions", []):
            if d.get("lang") == "en":
                txt = d.get("value","").strip()
                if txt and not txt.startswith("** REJECT"):
                    nvd_lookup[cve_id] = txt
                break
print(f"NVD loaded: {len(nvd_lookup):,}")

kev_cve_ids, kev_descs, kev_labels = [], [], []
for cve_id, val in kev_by_cve.items():
    if not val["techs"]:
        continue
    short = val["desc"]
    full  = nvd_lookup.get(cve_id, "")
    desc  = f"{short} {full}".strip() if full and len(full) > len(short) \
            else short
    if not desc:
        continue
    kev_cve_ids.append(cve_id)
    kev_descs.append(desc)
    kev_labels.append(val["techs"])

print(f"KEV eval rows: {len(kev_cve_ids)}")

# ── SMET ──────────────────────────────────────────────────────────
df_smet = pd.read_excel("CVE_annotated_dataset.xlsx", engine="openpyxl")

# name → T-code map
name_to_tc = {}
for tc, name in technique_names.items():
    name_to_tc[name.strip().lower()] = tc

# id2mitre fallback
try:
    import urllib.request
    with urllib.request.urlopen(
            "https://raw.githubusercontent.com/basel-a/SMET/main/id2mitre.json",
            timeout=10) as r:
        id2mitre = json.loads(r.read().decode("utf-8"))
    for k, v in id2mitre.items():
        if isinstance(v, str) and re.match(r'T\d{4}', v):
            parent_tc = parent_map.get(v, v)
            if parent_tc in technique_id_set:
                name_to_tc[k.strip().lower()] = parent_tc
    print(f"id2mitre loaded: {len(id2mitre)} entries")
except Exception as e:
    print(f"id2mitre fetch failed: {e}")

def parse_smet_techs(val):
    if pd.isna(val): return []
    try:    names = ast.literal_eval(str(val))
    except: names = [str(val)]
    return [name_to_tc[n.strip().lower()]
            for n in names if n.strip().lower() in name_to_tc]

df_smet["tcodes"] = df_smet["ATT&CK Techniques"].apply(parse_smet_techs)
df_smet_eval = df_smet[df_smet["tcodes"].apply(len) > 0].reset_index(drop=True)

smet_cve_ids = df_smet_eval["ID"].tolist()
smet_descs   = df_smet_eval["Description"].tolist()
smet_labels  = [set(tc) for tc in df_smet_eval["tcodes"].tolist()]

print(f"SMET eval rows: {len(df_smet_eval)}")

# ══════════════════════════════════════════════════════════════════
# B. LLM evaluation loop
# ══════════════════════════════════════════════════════════════════

def eval_llm(cve_ids, descs, labels, label, sample_n=None,
             use_db_hint=False, db=None, sleep_s=0.5):
    """
    Runs LLM on each CVE, computes R@1 / R@5 / R@10.
    sample_n: if set, randomly sample that many CVEs first.
    """
    if sample_n and sample_n < len(cve_ids):
        import random
        idx   = random.sample(range(len(cve_ids)), sample_n)
        cve_ids = [cve_ids[i] for i in idx]
        descs   = [descs[i]   for i in idx]
        labels  = [labels[i]  for i in idx]

    h1 = h5 = h10 = 0
    parse_failures = 0
    results = []

    for i, (cve_id, desc, true_labels) in enumerate(
            tqdm(zip(cve_ids, descs, labels),
                 total=len(cve_ids), desc=label)):

        db_hint = None
        if use_db_hint and db and cve_id in db:
            db_techs = db[cve_id].get("TECHNIQUES", [])
            if db_techs:
                names = [technique_names.get(tc, tc) for tc in db_techs]
                db_hint = ", ".join(names[:6])

        preds, raw = call_llm(desc, db_hint=db_hint)

        if not preds:
            parse_failures += 1

        hit1  = int(bool(preds) and preds[0] in true_labels)
        hit5  = int(bool(true_labels.intersection(set(preds[:5]))))
        hit10 = int(bool(true_labels.intersection(set(preds[:10]))))
        h1+=hit1; h5+=hit5; h10+=hit10

        results.append({
            "cve_id"      : cve_id,
            "predicted"   : preds,
            "true_labels" : list(true_labels),
            "hit1"        : hit1,
            "hit5"        : hit5,
            "hit10"       : hit10,
        })

        time.sleep(sleep_s)   # avoid hammering local server

    n = len(cve_ids)
    print(f"\n── {label} (n={n}) ──────────────────────────────────")
    print(f"  R@1  : {h1/n*100:.2f}%")
    print(f"  R@5  : {h5/n*100:.2f}%")
    print(f"  R@10 : {h10/n*100:.2f}%")
    print(f"  Parse failures: {parse_failures}/{n}")
    return {"r1":h1/n*100,"r5":h5/n*100,"r10":h10/n*100,
            "n":n, "failures":parse_failures, "rows":results}

# ── Load CVE2CAPEC DB for hint mode ───────────────────────────────
print("\nLoading CVE2CAPEC DB for hints...")
cve2capec_db = {}
for fpath in sorted(Path("OSRs/CVE2CAPEC").glob("*.jsonl")):
    with open(fpath, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                entry = json.loads(line)
            except: continue
            for cve_id, data in entry.items():
                raw_techs = data.get("TECHNIQUES", [])
                parent_tcs = set()
                for t in raw_techs:
                    t = t.strip()
                    if not t.upper().startswith("T"):
                        t = "T" + t
                    t = t.upper()
                    pt = parent_map.get(t, t)
                    if pt in technique_id_set:
                        parent_tcs.add(pt)
                cve2capec_db[cve_id] = {
                    "CWE"       : data.get("CWE", []),
                    "TECHNIQUES": list(parent_tcs),
                }
print(f"CVE2CAPEC loaded: {len(cve2capec_db):,} CVEs")

# ══════════════════════════════════════════════════════════════════
# C. Run evaluations
# (set sample_n=None to run full set — will take 20-40 min)
# ══════════════════════════════════════════════════════════════════
SAMPLE_N = None   # start with 50 per set to check speed/quality

print(f"\n{'='*60}")
print(f"LLM: {LM_MODEL}  |  sample_n={SAMPLE_N} per set")
print(f"{'='*60}")

# KEV — no hint
r_kev = eval_llm(kev_cve_ids, kev_descs, kev_labels,
                 f"KEV no hint", sample_n=SAMPLE_N)

# KEV — with DB hint
r_kev_hint = eval_llm(kev_cve_ids, kev_descs, kev_labels,
                      f"KEV + DB hint", sample_n=SAMPLE_N,
                      use_db_hint=True, db=cve2capec_db)

# SMET — no hint
r_smet = eval_llm(smet_cve_ids, smet_descs, smet_labels,
                  f"SMET no hint", sample_n=SAMPLE_N)

# SMET — with DB hint
r_smet_hint = eval_llm(smet_cve_ids, smet_descs, smet_labels,
                       f"SMET + DB hint", sample_n=SAMPLE_N,
                       use_db_hint=True, db=cve2capec_db)

# ══════════════════════════════════════════════════════════════════
# D. Comparison table
# ══════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("COMPARISON — LLM vs Best Previous Results")
print("="*70)
print(f"{'Method':<38} {'R@1':>8} {'R@5':>8} {'R@10':>8}")
print("─"*70)

rows = [
    # KEV
    ("KEV  | AttackBERT Rerank CE (best)",   13.13, 35.80, 54.65),
    ("KEV  | Gemma-4 no hint",
     r_kev['r1'],      r_kev['r5'],      r_kev['r10']),
    ("KEV  | Gemma-4 + DB hint",
     r_kev_hint['r1'], r_kev_hint['r5'], r_kev_hint['r10']),
    # SMET
    ("SMET | AttackBERT Hybrid RRF (best)",  32.45, 63.91, 76.16),
    ("SMET | Paper baseline (AttackBERT)",   32.45, 67.71, None),
    ("SMET | Gemma-4 no hint",
     r_smet['r1'],      r_smet['r5'],      r_smet['r10']),
    ("SMET | Gemma-4 + DB hint",
     r_smet_hint['r1'], r_smet_hint['r5'], r_smet_hint['r10']),
]

prev_set = None
for name, r1, r5, r10 in rows:
    cur = name[:4]
    if prev_set and prev_set != cur:
        print("─"*70)
    r10s = f"{r10:>7.2f}%" if r10 else "       —"
    print(f"{name:<38} {r1:>7.2f}% {r5:>7.2f}% {r10s}")
    prev_set = cur

print("="*70)
print(f"\nNote: LLM results on sample_n={SAMPLE_N} — "
      f"set SAMPLE_N=None for full run")

# ── Show a few examples ───────────────────────────────────────────
print("\n── Sample predictions (KEV, no hint) ───────────────────")
for row in r_kev["rows"][:3]:
    print(f"\n  CVE   : {row['cve_id']}")
    print(f"  Pred  : {row['predicted']}")
    print(f"  True  : {row['true_labels']}")
    print(f"  Hit@5 : {'✓' if row['hit5'] else '✗'}")

Loading NVD for KEV enrichment...
NVD loaded: 333,022
KEV eval rows: 419
id2mitre loaded: 594 entries
SMET eval rows: 302

Loading CVE2CAPEC DB for hints...
CVE2CAPEC loaded: 341,492 CVEs

LLM: gemma-4-26b-a4b  |  sample_n=None per set


KEV no hint: 100%|██████████| 419/419 [29:25<00:00,  4.21s/it]



── KEV no hint (n=419) ──────────────────────────────────
  R@1  : 51.79%
  R@5  : 78.28%
  R@10 : 78.28%
  Parse failures: 0/419


KEV + DB hint: 100%|██████████| 419/419 [29:37<00:00,  4.24s/it]



── KEV + DB hint (n=419) ──────────────────────────────────
  R@1  : 51.79%
  R@5  : 73.27%
  R@10 : 73.27%
  Parse failures: 0/419


SMET no hint: 100%|██████████| 302/302 [21:09<00:00,  4.20s/it]



── SMET no hint (n=302) ──────────────────────────────────
  R@1  : 37.75%
  R@5  : 74.17%
  R@10 : 74.17%
  Parse failures: 0/302


SMET + DB hint: 100%|██████████| 302/302 [21:32<00:00,  4.28s/it]


── SMET + DB hint (n=302) ──────────────────────────────────
  R@1  : 37.75%
  R@5  : 73.84%
  R@10 : 73.84%
  Parse failures: 0/302

COMPARISON — LLM vs Best Previous Results
Method                                      R@1      R@5     R@10
──────────────────────────────────────────────────────────────────────
KEV  | AttackBERT Rerank CE (best)       13.13%   35.80%   54.65%
KEV  | Gemma-4 no hint                   51.79%   78.28%   78.28%
KEV  | Gemma-4 + DB hint                 51.79%   73.27%   73.27%
──────────────────────────────────────────────────────────────────────
SMET | AttackBERT Hybrid RRF (best)      32.45%   63.91%   76.16%
SMET | Paper baseline (AttackBERT)       32.45%   67.71%        —
SMET | Gemma-4 no hint                   37.75%   74.17%   74.17%
SMET | Gemma-4 + DB hint                 37.75%   73.84%   73.84%

Note: LLM results on sample_n=None — set SAMPLE_N=None for full run

── Sample predictions (KEV, no hint) ───────────────────

  CVE   : CVE-2024-34102


# Ranker

In [ ]:
import pandas as pd
import ast
import json
import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModel
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Load ATT&CK Data
print("Loading ATT&CK STIX data...")
with open(r"OSRs\ATTACK\enterprise-attack-v16.1.json", "r", encoding="utf-8") as f:
    stix_bundle = json.load(f)

attack_dict = {}
name_to_tcode = {}
parent_map = {}

for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "attack-pattern" and not obj.get("revoked") and not obj.get("x_mitre_deprecated"):
        t_code = next((ref.get("external_id") for ref in obj.get("external_references", []) if ref.get("source_name") == "mitre-attack"), None)
        if t_code:
            attack_dict[t_code] = obj.get("description", "")
            name_to_tcode[obj.get("name")] = t_code

for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "relationship" and obj.get("relationship_type") == "subtechnique-of":
        sub_id, parent_id = obj.get("source_ref"), obj.get("target_ref")
        sub_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == sub_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        parent_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == parent_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        if sub_tcode and parent_tcode:
            parent_map[sub_tcode] = parent_tcode

parent_attack_dict = {k: v for k, v in attack_dict.items() if "." not in k}
technique_ids = list(parent_attack_dict.keys())
technique_texts = list(parent_attack_dict.values())

# 2. Load SMET Benchmark Data
print("Loading SMET benchmark data...")
smet_raw = pd.read_excel("CVE_annotated_dataset.xlsx")
with open("id2mitre.json", "r", encoding="utf-8") as f:
    id2mitre = json.load(f)

smet_records = []
for _, row in smet_raw.iterrows():
    try:
        tech_names = ast.literal_eval(row["ATT&CK Techniques"])
    except:
        tech_names = []
        
    t_codes = []
    for name in tech_names:
        mapped_id = next((k for k, v in id2mitre.items() if name in v), name_to_tcode.get(name))
        if mapped_id:
            t_codes.append(parent_map.get(mapped_id, mapped_id))
            
    if t_codes:
        smet_records.append({
            "Description": row["Description"],
            "T_Codes": list(set(t_codes))
        })
smet_df = pd.DataFrame(smet_records)
cve_texts = smet_df['Description'].tolist()
cve_ground_truths = smet_df['T_Codes'].tolist()

# 3. Initialize Models
print("Initializing AI Models...")
model_name = "basel/ATTACK-BERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device).eval()

def get_embeddings(text_list, batch_size=16):
    all_embeddings = []
    with torch.no_grad():
        for i in range(0, len(text_list), batch_size):
            batch = text_list[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
            output = model(**encoded)
            mask = encoded['attention_mask'].unsqueeze(-1).expand(output.last_hidden_state.size()).float()
            sum_emb = torch.sum(output.last_hidden_state * mask, 1)
            mean_pooled = F.normalize(sum_emb / torch.clamp(mask.sum(1), min=1e-9), p=2, dim=1)
            all_embeddings.append(mean_pooled.cpu())
    return torch.cat(all_embeddings, dim=0)

tech_embeddings = get_embeddings(technique_texts)
bm25_model = BM25Okapi([t.lower().split() for t in technique_texts])
print("Setup Complete.")

c:\Users\OA\Desktop\New Work\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Loading ATT&CK STIX data...
Loading SMET benchmark data...


Initializing AI Models...


Loading weights: 100%|██████████| 199/199 [00:00<?, ?it/s]
MPNetModel LOAD REPORT from: basel/ATTACK-BERT
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setup Complete.


In [ ]:
def map_threat_intel(description, top_k=5, rrf_k=60):
    """
    Takes raw threat intel or CVE descriptions, segments by attack vector,
    and returns top ATT&CK techniques using Hybrid RRF.
    """
    # 1. Chunking
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|;\s+', description) if len(s.strip()) > 10]
    if not sentences: sentences = [description]
        
    # 2. Semantic
    sentence_embs = get_embeddings(sentences)
    sim_matrix = torch.matmul(sentence_embs, tech_embeddings.T)
    semantic_scores = sim_matrix.max(dim=0).values.numpy()
    semantic_ranks = {idx: r + 1 for r, idx in enumerate(np.argsort(semantic_scores)[::-1])}
    
    # 3. Lexical
    bm25_scores = bm25_model.get_scores(description.lower().split())
    lexical_ranks = {idx: r + 1 for r, idx in enumerate(np.argsort(bm25_scores)[::-1])}
    
    # 4. RRF Fusion
    rrf_scores = {}
    for idx in range(len(technique_ids)):
        s_rank, l_rank = semantic_ranks[idx], lexical_ranks[idx]
        if bm25_scores[idx] == 0: l_rank = float('inf')
        rrf_scores[idx] = (1.0 / (rrf_k + s_rank)) + (1.0 / (rrf_k + l_rank))
        
    sorted_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    
    # 5. Output
    tcode_to_name = {v: k for k, v in name_to_tcode.items()}
    results = []
    for idx in sorted_indices[:top_k]:
        t_code = technique_ids[idx]
        results.append({
            "t_code": t_code,
            "name": tcode_to_name.get(t_code, "Unknown Technique"),
            "rrf_score": round(rrf_scores[idx], 4)
        })
    return results

# Test it!
sample_text = "Apache Log4j2 2.0-beta9 through 2.15.0 (excluding security releases 2.12.2, 2.12.3, and 2.3.1) JNDI features used in configuration, log messages, and parameters do not protect against attacker controlled LDAP and other JNDI related endpoints. An attacker who can control log messages or log message parameters can execute arbitrary code loaded from LDAP servers when message lookup substitution is enabled. From log4j 2.15.0, this behavior has been disabled by default. From version 2.16.0 (along with 2.12.2, 2.12.3, and 2.3.1), this functionality has been completely removed. Note that this vulnerability is specific to log4j-core and does not affect log4net, log4cxx, or other Apache Logging Services projects."
predictions = map_threat_intel(sample_text, top_k=5)

print(f"INPUT: {sample_text}\n")
for i, p in enumerate(predictions):
    print(f"#{i+1}: [{p['t_code']}] {p['name']} (Score: {p['rrf_score']})")

INPUT: Apache Log4j2 2.0-beta9 through 2.15.0 (excluding security releases 2.12.2, 2.12.3, and 2.3.1) JNDI features used in configuration, log messages, and parameters do not protect against attacker controlled LDAP and other JNDI related endpoints. An attacker who can control log messages or log message parameters can execute arbitrary code loaded from LDAP servers when message lookup substitution is enabled. From log4j 2.15.0, this behavior has been disabled by default. From version 2.16.0 (along with 2.12.2, 2.12.3, and 2.3.1), this functionality has been completely removed. Note that this vulnerability is specific to log4j-core and does not affect log4net, log4cxx, or other Apache Logging Services projects.

#1: [T1187] Forced Authentication (Score: 0.0313)
#2: [T1212] Exploitation for Credential Access (Score: 0.0286)
#3: [T1568] Dynamic Resolution (Score: 0.0263)
#4: [T1531] Account Access Removal (Score: 0.0259)
#5: [T1207] Rogue Domain Controller (Score: 0.0258)


In [ ]:
# ==============================================================================
# CELL 1: Setup & Data Extraction
# ==============================================================================
import pandas as pd
import ast
import json
import torch
import torch.nn.functional as F
import numpy as np
import pickle
import re
from transformers import AutoTokenizer, AutoModel
from rank_bm25 import BM25Okapi
from scipy.special import softmax
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Load ATT&CK STIX Data
print("Loading ATT&CK v16.1 Data...")
with open(r"OSRs\ATTACK\enterprise-attack-v16.1.json", "r", encoding="utf-8") as f:
    stix_bundle = json.load(f)

attack_dict, name_to_tcode, parent_map = {}, {}, {}

for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "attack-pattern" and not obj.get("revoked") and not obj.get("x_mitre_deprecated"):
        t_code = next((ref.get("external_id") for ref in obj.get("external_references", []) if ref.get("source_name") == "mitre-attack"), None)
        if t_code:
            attack_dict[t_code] = obj.get("description", "")
            name_to_tcode[obj.get("name")] = t_code

for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "relationship" and obj.get("relationship_type") == "subtechnique-of":
        sub_id = obj.get("source_ref")
        parent_id = obj.get("target_ref")
        sub_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == sub_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        parent_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == parent_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        if sub_tcode and parent_tcode: parent_map[sub_tcode] = parent_tcode

parent_attack_dict = {k: v for k, v in attack_dict.items() if "." not in k}
technique_ids = list(parent_attack_dict.keys())
technique_texts = list(parent_attack_dict.values())

# 2. Load SMET Benchmark
print("Loading SMET Benchmark...")
smet_raw = pd.read_excel("CVE_annotated_dataset.xlsx")
with open("id2mitre.json", "r", encoding="utf-8") as f: id2mitre = json.load(f)

smet_records = []
for _, row in smet_raw.iterrows():
    try: tech_names = ast.literal_eval(row["ATT&CK Techniques"])
    except: tech_names = []
        
    t_codes = []
    for name in tech_names:
        mapped_id = next((k for k, v in id2mitre.items() if name in v), name_to_tcode.get(name))
        if mapped_id: t_codes.append(parent_map.get(mapped_id, mapped_id))
            
    if t_codes:
        smet_records.append({"Description": row["Description"], "T_Codes": list(set(t_codes))})

smet_df = pd.DataFrame(smet_records)
cve_texts = smet_df['Description'].tolist()
cve_ground_truths = smet_df['T_Codes'].tolist()

print(f"Loaded {len(cve_texts)} valid SMET CVEs.")

c:\Users\OA\Desktop\New Work\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Loading ATT&CK v16.1 Data...
Loading SMET Benchmark...
Loaded 302 valid SMET CVEs.


In [ ]:
# ==============================================================================
# CELL 2: Initialize AI Models & Search Indices
# ==============================================================================
print("Initializing ATT&CK-BERT...")
model_name = "basel/ATTACK-BERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device).eval()

def get_embeddings(text_list, batch_size=16):
    all_embeddings = []
    with torch.no_grad():
        for i in range(0, len(text_list), batch_size):
            batch = text_list[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
            output = model(**encoded)
            mask = encoded['attention_mask'].unsqueeze(-1).expand(output.last_hidden_state.size()).float()
            sum_emb = torch.sum(output.last_hidden_state * mask, 1)
            mean_pooled = F.normalize(sum_emb / torch.clamp(mask.sum(1), min=1e-9), p=2, dim=1)
            all_embeddings.append(mean_pooled.cpu())
    return torch.cat(all_embeddings, dim=0)

print("Building Technique Embeddings...")
tech_embeddings = get_embeddings(technique_texts)

print("Building BM25 Lexical Index...")
bm25_model = BM25Okapi([t.lower().split() for t in technique_texts])

print("Loading SMET LR Classifier & Dictionaries...")
LR_model = pickle.load(open("LR_ATT&CK_model_V2.pkl", 'rb'))
with open('id2ATT&CK_V2.json', 'r', encoding='utf-8') as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

print("✅ All Models Ready.")

Initializing ATT&CK-BERT...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 34750.26it/s]
MPNetModel LOAD REPORT from: basel/ATTACK-BERT
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building Technique Embeddings...
Building BM25 Lexical Index...
Loading SMET LR Classifier & Dictionaries...
✅ All Models Ready.


c:\Users\OA\Desktop\New Work\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.3.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
import pandas as pd
import ast
import json
import re
import pickle
import numpy as np
from scipy.special import softmax
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from sklearn.metrics import coverage_error, label_ranking_loss, label_ranking_average_precision_score

print("Loading SMET Models and Classifiers...")

# 1. Load the Models exactly as SMET does
emb_model = SentenceTransformer("basel/ATTACK-BERT")
LR_model = pickle.load(open("LR_ATT&CK_model_V2.pkl", 'rb'))

# 2. Load the Label Dictionaries
id2mitre = json.load(open('id2mitre.json', 'r'))
id2label_raw = json.load(open('id2ATT&CK_V2.json', 'r'))
id2label = {int(i): v for i, v in id2label_raw.items()}

# ==========================================
# Benchmark Data Loading
# ==========================================
print("Loading ATT&CK STIX data & Benchmark Data...")
with open(r"OSRs\ATTACK\enterprise-attack-v16.1.json", "r", encoding="utf-8") as f:
    stix_bundle = json.load(f)

name_to_tcode = {}
parent_map = {}

for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "attack-pattern" and not obj.get("revoked") and not obj.get("x_mitre_deprecated"):
        t_code = next((ref.get("external_id") for ref in obj.get("external_references", []) if ref.get("source_name") == "mitre-attack"), None)
        if t_code:
            name_to_tcode[obj.get("name")] = t_code

for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "relationship" and obj.get("relationship_type") == "subtechnique-of":
        sub_id, parent_id = obj.get("source_ref"), obj.get("target_ref")
        sub_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == sub_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        parent_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == parent_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        if sub_tcode and parent_tcode:
            parent_map[sub_tcode] = parent_tcode

smet_raw = pd.read_excel("CVE_annotated_dataset.xlsx")
smet_records = []
for _, row in smet_raw.iterrows():
    try:
        tech_names = ast.literal_eval(row["ATT&CK Techniques"])
    except:
        tech_names = []
        
    t_codes = []
    for name in tech_names:
        mapped_id = next((k for k, v in id2mitre.items() if name in v), name_to_tcode.get(name))
        if mapped_id:
            t_codes.append(parent_map.get(mapped_id, mapped_id))
            
    if t_codes:
        smet_records.append({
            "Description": row["Description"],
            "T_Codes": list(set(t_codes))
        })
        
smet_df = pd.DataFrame(smet_records)
cve_texts = smet_df['Description'].tolist()
cve_ground_truths = smet_df['T_Codes'].tolist()

# ==============================================================================
# Setup Fixed Label Space for Matrix Creation
# ==============================================================================
# Extract all unique parent techniques that SMET is capable of predicting
all_smet_tcodes = list(id2label.values())
parent_classes = sorted(list(set([parent_map.get(t, t) for t in all_smet_tcodes])))
num_classes = len(parent_classes)

print(f"\nTotal Parent Classes handled by SMET model: {num_classes}")

# ==============================================================================
# The Exact SMET Prediction Logic
# ==============================================================================
def predict_techniques_smet(emb, clf, id2label_map, id2mitre_map):
    dec = clf.decision_function([emb])
    out = softmax(dec)[0]
    
    mapped_results = []
    for i in range(len(dec[0])):
        try:
            actual_t_code = id2label_map[i] 
            name_match = id2mitre_map[actual_t_code]
            actual_name = name_match[0] if isinstance(name_match, list) else name_match
            mapped_results.append((actual_t_code, actual_name, out[i]))
        except KeyError:
            continue
            
    return mapped_results # Removed sorting here for speed, handled at the end

def map_cve_smet_way(description): 
    # Notice: Removed top_k slice so we retain the full probability distribution!
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|;\s+', description) if len(s.strip()) > 10]
    attack_vectors = sentences + [description]
    
    all_predictions = []
    for av in attack_vectors:
        if not av.strip():
            continue
        emb = emb_model.encode(av)
        preds = predict_techniques_smet(emb, LR_model, id2label, id2mitre)
        all_predictions.extend(preds)
        
    consolidated = {}
    for t_code, name, prob in all_predictions:
        if t_code not in consolidated or prob > consolidated[t_code]['prob']:
            consolidated[t_code] = {'name': name, 'prob': prob}
            
    return consolidated

# ==========================================
# Full Dataset Evaluation Loop
# ==========================================
print("\nSetup Complete. Running full dataset evaluation...")

smet_metrics = {'R@1': [], 'R@5': [], 'R@10': []}
y_true_all = []
y_score_all = []
valid_cves = 0

for idx, cve_desc in enumerate(tqdm(cve_texts, desc="Evaluating Pure SMET")):
    ground_truth = set(cve_ground_truths[idx])
    n_true = len(ground_truth)
    
    if n_true == 0:
        continue
        
    valid_cves += 1

    # Get dictionary of ALL predictions
    smet_preds_dict = map_cve_smet_way(cve_desc)
    
    # 1. Roll up sub-technique probabilities to parent techniques
    parent_probs = {tid: 0.0 for tid in parent_classes}
    for t_code, data in smet_preds_dict.items():
        parent_tc = parent_map.get(t_code, t_code)
        if parent_tc in parent_probs:
            # If multiple sub-techniques share a parent, keep the highest probability
            parent_probs[parent_tc] = max(parent_probs[parent_tc], data['prob'])
            
    # 2. Populate Metric Matrices
    sample_y_true = np.zeros(num_classes)
    sample_y_score = np.zeros(num_classes)
    
    for i, tid in enumerate(parent_classes):
        sample_y_score[i] = parent_probs[tid]
        if tid in ground_truth:
            sample_y_true[i] = 1.0
            
    y_true_all.append(sample_y_true)
    y_score_all.append(sample_y_score)
    
    # 3. Calculate Recall @ K
    # Sort the parent indices by probability in descending order
    sorted_indices = np.argsort(sample_y_score)[::-1]
    sorted_tcodes = [parent_classes[i] for i in sorted_indices]
            
    for k in [1, 5, 10]:
        s_hits = len(set(sorted_tcodes[:k]).intersection(ground_truth))
        smet_metrics[f'R@{k}'].append(s_hits / n_true)

# Convert arrays for sklearn
y_true_all = np.array(y_true_all)
y_score_all = np.array(y_score_all)

# Calculate Advanced Metrics
lrap = label_ranking_average_precision_score(y_true_all, y_score_all)
rl = label_ranking_loss(y_true_all, y_score_all)
ce = coverage_error(y_true_all, y_score_all)

# Print Final Averaged Results
print("\n" + "="*50)
print(f"PURE SMET EVALUATION RESULTS (Tested on {valid_cves} CVEs)")
print("="*50)
print(f"Recall@1:       {(np.mean(smet_metrics['R@1']) * 100):.2f}%")
print(f"Recall@5:       {(np.mean(smet_metrics['R@5']) * 100):.2f}%")
print(f"Recall@10:      {(np.mean(smet_metrics['R@10']) * 100):.2f}%")
print("-" * 50)
print(f"LRAP:           {lrap * 100:.2f}%  (Higher is better)")
print(f"Ranking Loss:   {rl:.4f}   (Lower is better)")
print(f"Coverage Error: {ce:.4f}   (Lower is better)")
print("="*50)

Loading SMET Models and Classifiers...


Loading weights: 100%|██████████| 199/199 [00:00<?, ?it/s]
MPNetModel LOAD REPORT from: basel/ATTACK-BERT
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
c:\Users\OA\Desktop\New Work\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.3.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Loading ATT&CK STIX data & Benchmark Data...

Total Parent Classes handled by SMET model: 185

Setup Complete. Running full dataset evaluation...


Evaluating Pure SMET: 100%|██████████| 302/302 [00:20<00:00, 14.58it/s]



PURE SMET EVALUATION RESULTS (Tested on 302 CVEs)
Recall@1:       27.07%
Recall@5:       59.44%
Recall@10:      73.98%
--------------------------------------------------
LRAP:           45.93%  (Higher is better)
Ranking Loss:   0.0591   (Lower is better)
Coverage Error: 16.0232   (Lower is better)


In [ ]:
import numpy as np
import re
from tqdm.auto import tqdm
from scipy.special import softmax
from sklearn.metrics import coverage_error, label_ranking_loss, label_ranking_average_precision_score

print("\n" + "="*50)
print("PHASE 2: AFTER (SMET Classifier + BM25 Hybrid RRF Evaluation)")
print("="*50)

n_cves = len(cve_texts)
num_classes = len(technique_ids)

print(f"Total Candidate Techniques (Classes Handled): {num_classes}\n")

recall_at_1_sum = 0.0
recall_at_5_sum = 0.0
recall_at_10_sum = 0.0
valid_cves = 0

rrf_k = 60

# Arrays to store the full matrices for advanced metrics
y_true_all = []
y_score_all = []

for i in tqdm(range(n_cves), desc="Evaluating SMET Hybrid Pipeline"):
    true_labels = set(cve_ground_truths[i])
    n_true = len(true_labels)
    
    if n_true == 0:
        continue # Skip if no valid ground truth labels
        
    valid_cves += 1
    text = cve_texts[i]
    
    # 1. Chunking 
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|;\s+', text) if len(s.strip()) > 10]
    if not sentences: sentences = [text]
        
    # 2. Semantic Ranking (Using SMET SentenceTransformer + Logistic Regression)
    sentence_embs = emb_model.encode(sentences) 
    
    dec = LR_model.decision_function(sentence_embs)
    probs = softmax(dec, axis=1) # Shape: (num_chunks, num_classes_in_SMET)
    
    max_probs_per_class = np.max(probs, axis=0)
    
    # Roll up sub-technique probabilities to parent techniques
    parent_probs = {tid: 0.0 for tid in technique_ids}
    for class_idx, prob in enumerate(max_probs_per_class):
        try:
            t_code = id2label[class_idx]
            parent_tcode = parent_map.get(t_code, t_code)
            
            if parent_tcode in parent_probs:
                parent_probs[parent_tcode] = max(parent_probs[parent_tcode], prob)
        except KeyError:
            continue
            
    semantic_scores_aligned = [parent_probs[tid] for tid in technique_ids]
    semantic_ranks = {idx: r + 1 for r, idx in enumerate(np.argsort(semantic_scores_aligned)[::-1])}
    
    # 3. Lexical Ranking (BM25 on full text)
    bm25_scores = bm25_model.get_scores(text.lower().split())
    lexical_ranks = {idx: r + 1 for r, idx in enumerate(np.argsort(bm25_scores)[::-1])}
    
    # 4. Reciprocal Rank Fusion & Metric Matrix Building
    rrf_scores = {}
    
    # Arrays for this specific CVE to calculate LRAP, RL, and CE
    sample_y_true = np.zeros(num_classes)
    sample_y_score = np.zeros(num_classes)
    
    for idx in range(num_classes):
        s_rank, l_rank = semantic_ranks[idx], lexical_ranks[idx]
        if bm25_scores[idx] == 0: l_rank = float('inf') 
        
        # Calculate RRF Score
        fused_score = (1.0 / (rrf_k + s_rank)) + (1.0 / (rrf_k + l_rank))
        rrf_scores[idx] = fused_score
        
        # Populate metric matrices
        sample_y_score[idx] = fused_score
        if technique_ids[idx] in true_labels:
            sample_y_true[idx] = 1.0
            
    y_true_all.append(sample_y_true)
    y_score_all.append(sample_y_score)
        
    # Retrieve top predictions
    sorted_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    top_10_preds = [technique_ids[idx] for idx in sorted_indices[:10]]
    
    # R@1
    hits_1 = len(true_labels.intersection(set(top_10_preds[:1])))
    recall_at_1_sum += hits_1 / n_true
    
    # R@5
    hits_5 = len(true_labels.intersection(set(top_10_preds[:5])))
    recall_at_5_sum += hits_5 / n_true
    
    # R@10
    hits_10 = len(true_labels.intersection(set(top_10_preds[:10])))
    recall_at_10_sum += hits_10 / n_true

# Convert to numpy arrays for sklearn metric functions
y_true_all = np.array(y_true_all)
y_score_all = np.array(y_score_all)

# Calculate Advanced Metrics
lrap = label_ranking_average_precision_score(y_true_all, y_score_all)
rl = label_ranking_loss(y_true_all, y_score_all)
ce = coverage_error(y_true_all, y_score_all)

print("\n" + "="*50)
print(f"HYBRID SMET RESULTS (Evaluated on {valid_cves} CVEs)")
print("="*50)
print(f"Recall@1:       {(recall_at_1_sum / valid_cves) * 100:.2f}%")
print(f"Recall@5:       {(recall_at_5_sum / valid_cves) * 100:.2f}%")
print(f"Recall@10:      {(recall_at_10_sum / valid_cves) * 100:.2f}%")
print("-" * 50)
print(f"LRAP:           {lrap * 100:.2f}%  (Higher is better)")
print(f"Ranking Loss:   {rl:.4f}   (Lower is better)")
print(f"Coverage Error: {ce:.4f}   (Lower is better)")
print("="*50)


PHASE 2: AFTER (SMET Classifier + BM25 Hybrid RRF Evaluation)
Total Candidate Techniques (Classes Handled): 203



Evaluating SMET Hybrid Pipeline: 100%|██████████| 302/302 [00:07<00:00, 42.00it/s]



HYBRID SMET RESULTS (Evaluated on 302 CVEs)
Recall@1:       30.88%
Recall@5:       63.08%
Recall@10:      73.57%
--------------------------------------------------
LRAP:           49.11%  (Higher is better)
Ranking Loss:   0.0421   (Lower is better)
Coverage Error: 12.4603   (Lower is better)
